In [ ]:
# Notebook Automatizado - Mapas de Calor de Recuperaciones (Versión Adaptable)
# ==============================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mplsoccer import Pitch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURACIÓN GLOBAL
# =====================================================================

# ACTUALIZA ESTA RUTA según lo que encuentre el diagnóstico
BASE_DIR = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\matchcenter\MatchCenter\Competition\Season")
SAVE_PATH = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\assets\viz")

# Configuración visual
PITCH_COLOR = "#24282a"
LINE_COLOR = "white"
flamingo_cmap = LinearSegmentedColormap.from_list("Flamingo - 10 colors", ['#e3aca7', '#c03a1d'], N=10)

# Crear directorio de salida si no existe
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# =====================================================================
# FUNCIONES DE DETECCIÓN AUTOMÁTICA DE ESTRUCTURA
# =====================================================================

def find_match_directories(base_path, max_depth=5):
    """
    Busca directorios que contienen archivos CSV de partidos.
    Detecta automáticamente la estructura sin importar la profundidad.
    """
    match_dirs = []
    
    def explore_recursive(path, current_depth=0):
        if current_depth > max_depth or not path.is_dir():
            return
            
        try:
            # Buscar archivos CSV en este directorio
            csv_files = [f for f in path.iterdir() if f.is_file() and f.suffix.lower() == '.csv']
            
            # Si encuentra CSVs, verificar si son archivos de partidos
            if csv_files:
                csv_names = [f.name.lower() for f in csv_files]
                
                # Buscar patrones que indiquen archivos de eventos/jugadores
                has_events = any('event' in name or 'defensive' in name for name in csv_names)
                has_players = any('player' in name for name in csv_names)
                
                if has_events or has_players or len(csv_files) > 3:  # Criterio flexible
                    match_dirs.append(path)
                    print(f"✅ Directorio de partido encontrado: {path}")
                    print(f"   Archivos CSV: {[f.name for f in csv_files[:5]]}")
            
            # Continuar explorando subdirectorios
            for subdir in path.iterdir():
                if subdir.is_dir():
                    explore_recursive(subdir, current_depth + 1)
                    
        except (PermissionError, OSError) as e:
            print(f"⚠️  Sin acceso a {path}: {e}")
    
    print(f"🔍 Buscando directorios de partidos en: {base_path}")
    explore_recursive(base_path)
    
    return match_dirs

def smart_find_csv(directory, patterns):
    """
    Busca archivos CSV usando múltiples patrones de nombres.
    """
    if not directory.is_dir():
        return None
    
    for pattern in patterns:
        # Búsqueda exacta
        exact_file = directory / f"{pattern}.csv"
        if exact_file.exists():
            return exact_file
        
        # Búsqueda por contenido en el nombre
        for file in directory.iterdir():
            if (file.is_file() and 
                file.suffix.lower() == '.csv' and 
                pattern.lower() in file.name.lower()):
                return file
    
    return None

def smart_read_players_csv(csv_path):
    """
    Lee un CSV de jugadores y trata de normalizar las columnas automáticamente.
    """
    try:
        df = pd.read_csv(csv_path)
        print(f"📊 Leyendo jugadores: {csv_path.name} ({df.shape[0]} filas, {df.shape[1]} columnas)")
        
        # Mostrar columnas disponibles
        print(f"   Columnas: {list(df.columns)}")
        
        # Mapeo inteligente de columnas
        col_mapping = {}
        cols_lower = {col.lower().strip(): col for col in df.columns}
        
        # Buscar ID del jugador
        for pattern in ['player_id', 'playerid', 'id', 'player id']:
            if pattern in cols_lower:
                col_mapping['player_id'] = cols_lower[pattern]
                break
        
        # Buscar nombre del jugador
        for pattern in ['player_name', 'name', 'player', 'playername', 'player name']:
            if pattern in cols_lower:
                col_mapping['player_name'] = cols_lower[pattern]
                break
        
        # Buscar ID del equipo
        for pattern in ['team_id', 'teamid', 'team id']:
            if pattern in cols_lower:
                col_mapping['team_id'] = cols_lower[pattern]
                break
        
        # Buscar nombre del equipo
        for pattern in ['team_name', 'team', 'teamname', 'team name']:
            if pattern in cols_lower:
                col_mapping['team_name'] = cols_lower[pattern]
                break
        
        print(f"   Mapeo detectado: {col_mapping}")
        
        if 'player_id' not in col_mapping or 'player_name' not in col_mapping:
            print(f"⚠️  Columnas esenciales no encontradas en {csv_path.name}")
            return None
        
        # Crear DataFrame normalizado
        result = pd.DataFrame()
        result['player_id'] = pd.to_numeric(df[col_mapping['player_id']], errors='coerce')
        result['player_name'] = df[col_mapping['player_name']].astype(str)
        
        if 'team_id' in col_mapping:
            result['team_id'] = pd.to_numeric(df[col_mapping['team_id']], errors='coerce')
        
        if 'team_name' in col_mapping:
            result['team_name'] = df[col_mapping['team_name']].astype(str)
        
        # Limpiar datos
        result = result.dropna(subset=['player_id'])
        result['player_id'] = result['player_id'].astype(int)
        
        return result
        
    except Exception as e:
        print(f"❌ Error leyendo {csv_path}: {e}")
        return None

def smart_read_events_csv(csv_path):
    """
    Lee un CSV de eventos y trata de normalizar las columnas automáticamente.
    """
    try:
        df = pd.read_csv(csv_path)
        print(f"📊 Leyendo eventos: {csv_path.name} ({df.shape[0]} filas, {df.shape[1]} columnas)")
        
        # Mostrar muestra de datos
        print(f"   Columnas: {list(df.columns)}")
        if len(df) > 0:
            print(f"   Muestra: {df.iloc[0].to_dict()}")
        
        # Mapeo inteligente de columnas
        col_mapping = {}
        cols_lower = {col.lower().strip(): col for col in df.columns}
        
        # Coordenadas
        for pattern in ['x']:
            if pattern in cols_lower:
                col_mapping['x'] = cols_lower[pattern]
                break
        
        for pattern in ['y']:
            if pattern in cols_lower:
                col_mapping['y'] = cols_lower[pattern]
                break
        
        # ID del jugador
        for pattern in ['playerid', 'player_id', 'player id']:
            if pattern in cols_lower:
                col_mapping['playerId'] = cols_lower[pattern]
                break
        
        # Tipo de evento
        for pattern in ['typename', 'type', 'event', 'eventname', 'type name', 'event name']:
            if pattern in cols_lower:
                col_mapping['typeName'] = cols_lower[pattern]
                break
        
        # ID del equipo
        for pattern in ['teamid', 'team_id', 'team id']:
            if pattern in cols_lower:
                col_mapping['teamId'] = cols_lower[pattern]
                break
        
        print(f"   Mapeo detectado: {col_mapping}")
        
        # Verificar columnas esenciales
        required = ['x', 'y', 'playerId']
        missing = [col for col in required if col not in col_mapping]
        
        if missing:
            print(f"⚠️  Columnas esenciales faltantes: {missing}")
            return None
        
        # Crear DataFrame normalizado
        result = pd.DataFrame()
        result['x'] = pd.to_numeric(df[col_mapping['x']], errors='coerce')
        result['y'] = pd.to_numeric(df[col_mapping['y']], errors='coerce')
        result['playerId'] = pd.to_numeric(df[col_mapping['playerId']], errors='coerce')
        
        if 'typeName' in col_mapping:
            result['typeName'] = df[col_mapping['typeName']].astype(str)
        
        if 'teamId' in col_mapping:
            result['teamId'] = pd.to_numeric(df[col_mapping['teamId']], errors='coerce')
        
        # Limpiar datos
        result = result.dropna(subset=['x', 'y', 'playerId'])
        
        return result
        
    except Exception as e:
        print(f"❌ Error leyendo {csv_path}: {e}")
        return None

# =====================================================================
# FUNCIÓN PRINCIPAL DE RECOLECCIÓN (MEJORADA)
# =====================================================================

def collect_ball_recoveries_adaptive(base_dir):
    """
    Versión adaptable que detecta automáticamente la estructura de directorios.
    """
    print(f"🚀 Iniciando recolección adaptable desde: {base_dir}")
    
    # Buscar directorios de partidos automáticamente
    match_dirs = find_match_directories(base_dir)
    
    if not match_dirs:
        raise RuntimeError(f"No se encontraron directorios con archivos CSV en {base_dir}")
    
    print(f"📁 Encontrados {len(match_dirs)} directorios de partidos")
    
    all_recoveries = []
    all_players = []
    
    for i, match_dir in enumerate(match_dirs):
        print(f"\n📂 Procesando {i+1}/{len(match_dirs)}: {match_dir.name}")
        
        # Buscar archivo de jugadores con múltiples patrones
        players_patterns = ['players', 'player', 'roster', 'lineup']
        players_file = smart_find_csv(match_dir, players_patterns)
        
        if not players_file:
            print(f"   ⚠️  No se encontró archivo de jugadores")
            continue
        
        players_df = smart_read_players_csv(players_file)
        if players_df is None:
            continue
        
        # Buscar archivo de eventos con múltiples patrones
        events_patterns = ['events_defensive', 'defensive', 'events', 'event', 'actions']
        events_file = smart_find_csv(match_dir, events_patterns)
        
        if not events_file:
            print(f"   ⚠️  No se encontró archivo de eventos")
            continue
        
        events_df = smart_read_events_csv(events_file)
        if events_df is None:
            continue
        
        # Filtrar recuperaciones (con múltiples patrones)
        if 'typeName' in events_df.columns:
            recovery_patterns = ['ballrecovery', 'ball recovery', 'recovery', 'recuperacion']
            recovery_mask = events_df['typeName'].str.lower().isin(recovery_patterns)
            recoveries = events_df[recovery_mask]
        else:
            print(f"   ⚠️  No se encontró columna de tipo de evento, usando todos los eventos")
            recoveries = events_df
        
        if recoveries.empty:
            print(f"   ⚠️  No se encontraron recuperaciones")
            continue
        
        # Añadir información de jugadores
        recoveries_with_players = recoveries.merge(
            players_df, 
            left_on='playerId', 
            right_on='player_id', 
            how='left'
        )
        
        all_recoveries.append(recoveries_with_players)
        all_players.append(players_df)
        
        print(f"   ✅ {len(recoveries)} recuperaciones procesadas")
    
    if not all_recoveries:
        raise RuntimeError("No se encontraron recuperaciones en ningún partido")
    
    # Combinar todos los datos
    combined_recoveries = pd.concat(all_recoveries, ignore_index=True)
    combined_players = pd.concat(all_players, ignore_index=True).drop_duplicates(subset=['player_id'])
    
    # Crear ranking
    ranking = (combined_recoveries
               .groupby(['player_id', 'player_name'])
               .agg({
                   'playerId': 'count',
                   'team_name': 'first'
               })
               .rename(columns={'playerId': 'recoveries'})
               .reset_index()
               .sort_values('recoveries', ascending=False))
    
    print(f"\n✅ Procesamiento completado:")
    print(f"   📊 Total recuperaciones: {len(combined_recoveries)}")
    print(f"   👥 Jugadores únicos: {len(ranking)}")
    
    return combined_recoveries, ranking

# =====================================================================
# FUNCIONES DE VISUALIZACIÓN (ADAPTADAS)
# =====================================================================

def plot_individual_recovery_heatmap_adaptive(recoveries_df, player_name, save_image=True, show_plot=True, figsize=(10, 6)):
    """
    Versión adaptada de la función de mapa individual con KDE suave.
    """
    # Buscar jugador
    player_mask = recoveries_df['player_name'].str.lower() == player_name.lower()
    player_data = recoveries_df[player_mask]
    
    if player_data.empty:
        available = recoveries_df['player_name'].value_counts().head(10)
        raise ValueError(f"Jugador '{player_name}' no encontrado.\n"
                        f"Jugadores disponibles (top 10):\n{available}")
    
    # Información del jugador
    player_info = player_data.iloc[0]
    team_name = player_info.get('team_name', 'Unknown')
    total_recoveries = len(player_data)
    
    # Crear visualización con campo horizontal
    pitch = Pitch(pitch_type='opta', pitch_color=PITCH_COLOR, line_color=LINE_COLOR, 
                 axis=False)
    fig, ax = pitch.draw(figsize=figsize)
    
    # Usar KDE para mapa de calor suave (como en tu ejemplo)
    if len(player_data) >= 5:  # Necesitamos suficientes puntos para KDE
        pitch.kdeplot(player_data['x'], player_data['y'], ax=ax, 
                     cmap=flamingo_cmap, fill=True, levels=15, alpha=0.7,
                     thresh=0.05)
    else:
        # Si hay pocos puntos, usar scatter
        pitch.scatter(player_data['x'], player_data['y'], ax=ax, 
                     s=200, c='#c03a1d', alpha=0.8, edgecolors='white', linewidth=1)
    
    # Título más elegante
    title = f"{player_name} ({team_name})\nRecuperaciones de Balón (n={total_recoveries})"
    ax.set_title(title, color=LINE_COLOR, fontsize=14, pad=20, 
                fontweight='bold')
    fig.patch.set_facecolor(PITCH_COLOR)
    
    # Guardar
    if save_image:
        safe_name = player_name.replace(" ", "_").replace(".", "")
        filename = f"recuperaciones_{safe_name}.png"
        filepath = SAVE_PATH / filename
        plt.savefig(filepath, dpi=300, bbox_inches='tight', 
                   facecolor=PITCH_COLOR, edgecolor='none')
        print(f"💾 Imagen guardada: {filepath}")
    
    if show_plot:
        plt.show()
    
    return fig, ax

def plot_top3_recovery_heatmaps_adaptive(recoveries_df, ranking_df, save_image=True, show_plot=True):
    """
    Versión adaptada para TOP 3 con KDE suave.
    """
    top3 = ranking_df.head(3)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor(PITCH_COLOR)
    
    for i, (_, player) in enumerate(top3.iterrows()):
        player_id = player['player_id']
        player_name = player['player_name']
        team_name = player.get('team_name', 'Unknown')
        recoveries_count = player['recoveries']
        
        # Filtrar datos del jugador
        player_data = recoveries_df[recoveries_df['player_id'] == player_id]
        
        # Crear pitch
        pitch = Pitch(pitch_type='opta', pitch_color=PITCH_COLOR, 
                     line_color=LINE_COLOR, axis=False)
        pitch.draw(ax=axes[i])
        
        # Mapa de calor suave con KDE
        if not player_data.empty and len(player_data) >= 5:
            pitch.kdeplot(player_data['x'], player_data['y'], ax=axes[i], 
                         cmap=flamingo_cmap, fill=True, levels=12, alpha=0.7,
                         thresh=0.05)
        elif not player_data.empty:
            # Scatter para pocos puntos
            pitch.scatter(player_data['x'], player_data['y'], ax=axes[i], 
                         s=150, c='#c03a1d', alpha=0.8, 
                         edgecolors='white', linewidth=1)
        
        # Título más elegante
        title = f"#{i+1} {player_name}\n({team_name})\n{recoveries_count} recuperaciones"
        axes[i].set_title(title, color=LINE_COLOR, fontsize=11, pad=15,
                         fontweight='bold')
    
    # Título general más elegante
    fig.suptitle("TOP 3 - Jugadores con Más Recuperaciones de Balón", 
                color=LINE_COLOR, fontsize=16, y=0.95, fontweight='bold')
    plt.tight_layout()
    
    if save_image:
        filepath = SAVE_PATH / "top3_recuperaciones.png"
        plt.savefig(filepath, dpi=300, bbox_inches='tight', 
                   facecolor=PITCH_COLOR, edgecolor='none')
        print(f"💾 TOP 3 guardado: {filepath}")
    
    if show_plot:
        plt.show()
    
    return fig

# =====================================================================
# FUNCIÓN PRINCIPAL ADAPTADA
# =====================================================================

def main_adaptive():
    """
    Función principal que se adapta automáticamente a la estructura de datos.
    """
    try:
        print("🚀 Iniciando análisis adaptativo de recuperaciones...")
        print("=" * 60)
        
        # Verificar que el directorio base existe
        if not BASE_DIR.exists():
            print(f"❌ El directorio base no existe: {BASE_DIR}")
            print("Por favor, actualiza BASE_DIR con la ruta correcta")
            return None, None
        
        # Recopilar datos de forma adaptativa
        recoveries_df, ranking_df = collect_ball_recoveries_adaptive(BASE_DIR)
        
        # Estadísticas
        print(f"\n📈 Estadísticas:")
        print(f"   📊 Total recuperaciones: {len(recoveries_df):,}")
        print(f"   👥 Jugadores únicos: {len(ranking_df):,}")
        
        # Mostrar TOP 5
        print(f"\n🏆 TOP 5 Jugadores:")
        for i, (_, player) in enumerate(ranking_df.head(5).iterrows()):
            team = player.get('team_name', 'Unknown')
            print(f"   {i+1}. {player['player_name']} ({team}) - {player['recoveries']} recuperaciones")
        
        # Generar TOP 3
        print(f"\n🎨 Generando TOP 3...")
        plot_top3_recovery_heatmaps_adaptive(recoveries_df, ranking_df)
        
        print(f"\n✅ ¡Completado exitosamente!")
        return recoveries_df, ranking_df
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\n💡 Sugerencias:")
        print("1. Ejecuta el código de diagnóstico primero")
        print("2. Verifica que BASE_DIR apunte al directorio correcto")
        print("3. Asegúrate de que los archivos CSV existen")
        raise

# =====================================================================
# EJECUCIÓN
# =====================================================================

if __name__ == "__main__":
    # Ejecutar versión adaptativa
    recoveries_df, ranking_df = main_adaptive()
    
    # Ejemplo de uso individual
    if recoveries_df is not None and not ranking_df.empty:
        top_player = ranking_df.iloc[0]['player_name']
        print(f"\n🎯 Ejemplo - Mapa individual para: {top_player}")
        plot_individual_recovery_heatmap_adaptive(recoveries_df, top_player)

print("✅ Notebook adaptativo listo!")

In [ ]:
plot_individual_recovery_heatmap_adaptive(recoveries_df, 'Pablo Fornals')

In [ ]:
# Notebook - Mapas de Calor de Acciones Defensivas (Tackles + Intercepciones)
# =============================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mplsoccer import Pitch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURACIÓN GLOBAL
# =====================================================================

BASE_DIR = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\matchcenter\MatchCenter\Competition\Season")
SAVE_PATH = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\assets\viz")

# Configuración visual
PITCH_COLOR = "#24282a"
LINE_COLOR = "white"
flamingo_cmap = LinearSegmentedColormap.from_list("Flamingo - 10 colors", ['#e3aca7', '#c03a1d'], N=10)

# Crear directorio de salida si no existe
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# =====================================================================
# FUNCIONES DE DETECCIÓN PARA ACCIONES DEFENSIVAS
# =====================================================================

def find_match_directories(base_path, max_depth=5):
    """
    Busca directorios que contienen archivos CSV de partidos.
    """
    match_dirs = []
    
    def explore_recursive(path, current_depth=0):
        if current_depth > max_depth or not path.is_dir():
            return
            
        try:
            csv_files = [f for f in path.iterdir() if f.is_file() and f.suffix.lower() == '.csv']
            
            if csv_files:
                csv_names = [f.name.lower() for f in csv_files]
                has_events = any('event' in name or 'defensive' in name for name in csv_names)
                has_players = any('player' in name for name in csv_names)
                
                if has_events or has_players or len(csv_files) > 3:
                    match_dirs.append(path)
                    print(f"✅ Directorio encontrado: {path}")
                    print(f"   Archivos CSV: {[f.name for f in csv_files[:5]]}")
            
            for subdir in path.iterdir():
                if subdir.is_dir():
                    explore_recursive(subdir, current_depth + 1)
                    
        except (PermissionError, OSError) as e:
            print(f"⚠️  Sin acceso a {path}: {e}")
    
    print(f"🔍 Buscando directorios de partidos en: {base_path}")
    explore_recursive(base_path)
    return match_dirs

def smart_find_csv(directory, patterns):
    """Busca archivos CSV usando múltiples patrones de nombres."""
    if not directory.is_dir():
        return None
    
    for pattern in patterns:
        exact_file = directory / f"{pattern}.csv"
        if exact_file.exists():
            return exact_file
        
        for file in directory.iterdir():
            if (file.is_file() and 
                file.suffix.lower() == '.csv' and 
                pattern.lower() in file.name.lower()):
                return file
    return None

def smart_read_players_csv(csv_path):
    """Lee y normaliza CSV de jugadores."""
    try:
        df = pd.read_csv(csv_path)
        print(f"📊 Leyendo jugadores: {csv_path.name} ({df.shape[0]} filas)")
        
        col_mapping = {}
        cols_lower = {col.lower().strip(): col for col in df.columns}
        
        # Mapear columnas esenciales
        for pattern in ['player_id', 'playerid', 'id', 'player id']:
            if pattern in cols_lower:
                col_mapping['player_id'] = cols_lower[pattern]
                break
        
        for pattern in ['player_name', 'name', 'player', 'playername', 'player name']:
            if pattern in cols_lower:
                col_mapping['player_name'] = cols_lower[pattern]
                break
        
        for pattern in ['team_id', 'teamid', 'team id']:
            if pattern in cols_lower:
                col_mapping['team_id'] = cols_lower[pattern]
                break
        
        for pattern in ['team_name', 'team', 'teamname', 'team name']:
            if pattern in cols_lower:
                col_mapping['team_name'] = cols_lower[pattern]
                break
        
        if 'player_id' not in col_mapping or 'player_name' not in col_mapping:
            print(f"⚠️  Columnas esenciales no encontradas")
            return None
        
        result = pd.DataFrame()
        result['player_id'] = pd.to_numeric(df[col_mapping['player_id']], errors='coerce')
        result['player_name'] = df[col_mapping['player_name']].astype(str)
        
        if 'team_id' in col_mapping:
            result['team_id'] = pd.to_numeric(df[col_mapping['team_id']], errors='coerce')
        if 'team_name' in col_mapping:
            result['team_name'] = df[col_mapping['team_name']].astype(str)
        
        result = result.dropna(subset=['player_id'])
        result['player_id'] = result['player_id'].astype(int)
        
        return result
        
    except Exception as e:
        print(f"❌ Error leyendo {csv_path}: {e}")
        return None

def smart_read_defensive_events(csv_path):
    """
    Lee eventos defensivos y filtra TACKLES e INTERCEPCIONES.
    """
    try:
        df = pd.read_csv(csv_path)
        print(f"📊 Leyendo eventos defensivos: {csv_path.name} ({df.shape[0]} filas)")
        
        col_mapping = {}
        cols_lower = {col.lower().strip(): col for col in df.columns}
        
        # Mapear columnas
        for pattern in ['x']:
            if pattern in cols_lower:
                col_mapping['x'] = cols_lower[pattern]
                break
        
        for pattern in ['y']:
            if pattern in cols_lower:
                col_mapping['y'] = cols_lower[pattern]
                break
        
        for pattern in ['playerid', 'player_id', 'player id']:
            if pattern in cols_lower:
                col_mapping['playerId'] = cols_lower[pattern]
                break
        
        for pattern in ['typename', 'type', 'event', 'eventname', 'type name']:
            if pattern in cols_lower:
                col_mapping['typeName'] = cols_lower[pattern]
                break
        
        for pattern in ['teamid', 'team_id', 'team id']:
            if pattern in cols_lower:
                col_mapping['teamId'] = cols_lower[pattern]
                break
        
        required = ['x', 'y', 'playerId']
        missing = [col for col in required if col not in col_mapping]
        
        if missing:
            print(f"⚠️  Columnas faltantes: {missing}")
            return None
        
        result = pd.DataFrame()
        result['x'] = pd.to_numeric(df[col_mapping['x']], errors='coerce')
        result['y'] = pd.to_numeric(df[col_mapping['y']], errors='coerce')
        result['playerId'] = pd.to_numeric(df[col_mapping['playerId']], errors='coerce')
        
        if 'typeName' in col_mapping:
            result['typeName'] = df[col_mapping['typeName']].astype(str)
        if 'teamId' in col_mapping:
            result['teamId'] = pd.to_numeric(df[col_mapping['teamId']], errors='coerce')
        
        result = result.dropna(subset=['x', 'y', 'playerId'])
        
        # FILTRAR TACKLES E INTERCEPCIONES
        if 'typeName' in result.columns:
            defensive_patterns = [
                'tacklewon', 'tackle won', 'tackle', 
                'interceptionwon', 'interception won', 'interception',
                'ballrecovery', 'ball recovery', 'recovery'
            ]
            
            mask = result['typeName'].str.lower().isin(defensive_patterns)
            result = result[mask]
            
            print(f"   ✅ {len(result)} acciones defensivas encontradas")
            
            # Mostrar tipos de eventos encontrados
            event_types = result['typeName'].value_counts()
            print(f"   📋 Tipos de eventos: {dict(event_types)}")
        
        return result
        
    except Exception as e:
        print(f"❌ Error leyendo {csv_path}: {e}")
        return None

# =====================================================================
# FUNCIÓN PRINCIPAL DE RECOLECCIÓN
# =====================================================================

def collect_defensive_actions(base_dir):
    """
    Recopila tackles, intercepciones y recuperaciones de todos los partidos.
    """
    print(f"🚀 Recopilando acciones defensivas desde: {base_dir}")
    
    match_dirs = find_match_directories(base_dir)
    
    if not match_dirs:
        raise RuntimeError(f"No se encontraron directorios con archivos CSV")
    
    print(f"📁 Procesando {len(match_dirs)} directorios de partidos")
    
    all_actions = []
    
    for i, match_dir in enumerate(match_dirs):
        print(f"\n📂 Procesando {i+1}/{len(match_dirs)}: {match_dir.name}")
        
        # Buscar jugadores
        players_patterns = ['players', 'player', 'roster', 'lineup']
        players_file = smart_find_csv(match_dir, players_patterns)
        
        if not players_file:
            print(f"   ⚠️  No se encontró archivo de jugadores")
            continue
        
        players_df = smart_read_players_csv(players_file)
        if players_df is None:
            continue
        
        # Buscar eventos defensivos
        events_patterns = ['events_defensive', 'defensive', 'events', 'event', 'actions']
        events_file = smart_find_csv(match_dir, events_patterns)
        
        if not events_file:
            print(f"   ⚠️  No se encontró archivo de eventos")
            continue
        
        actions_df = smart_read_defensive_events(events_file)
        if actions_df is None:
            continue
        
        if actions_df.empty:
            print(f"   ⚠️  No se encontraron acciones defensivas")
            continue
        
        # Añadir información de jugadores
        actions_with_players = actions_df.merge(
            players_df, 
            left_on='playerId', 
            right_on='player_id', 
            how='left'
        )
        
        all_actions.append(actions_with_players)
        print(f"   ✅ {len(actions_df)} acciones defensivas procesadas")
    
    if not all_actions:
        raise RuntimeError("No se encontraron acciones defensivas en ningún partido")
    
    # Combinar datos
    combined_actions = pd.concat(all_actions, ignore_index=True)
    
    # Crear ranking
    ranking = (combined_actions
               .groupby(['player_id', 'player_name'])
               .agg({
                   'playerId': 'count',
                   'team_name': 'first',
                   'typeName': lambda x: dict(x.value_counts())  # Contar tipos de acciones
               })
               .rename(columns={'playerId': 'total_actions'})
               .reset_index()
               .sort_values('total_actions', ascending=False))
    
    print(f"\n✅ Procesamiento completado:")
    print(f"   📊 Total acciones defensivas: {len(combined_actions)}")
    print(f"   👥 Jugadores únicos: {len(ranking)}")
    
    return combined_actions, ranking

# =====================================================================
# FUNCIONES DE VISUALIZACIÓN
# =====================================================================

def plot_defensive_actions_heatmap(actions_df, player_name, save_image=True, show_plot=True, figsize=(10, 6)):
    """
    Genera mapa de calor de acciones defensivas (tackles + intercepciones + recuperaciones).
    """
    player_mask = actions_df['player_name'].str.lower() == player_name.lower()
    player_data = actions_df[player_mask]
    
    if player_data.empty:
        available = actions_df['player_name'].value_counts().head(10)
        raise ValueError(f"Jugador '{player_name}' no encontrado.\n"
                        f"Jugadores disponibles (top 10):\n{available}")
    
    # Información del jugador
    player_info = player_data.iloc[0]
    team_name = player_info.get('team_name', 'Unknown')
    total_actions = len(player_data)
    
    # Contar tipos de acciones
    action_counts = player_data['typeName'].value_counts().to_dict()
    tackles = sum(v for k, v in action_counts.items() if 'tackle' in k.lower())
    interceptions = sum(v for k, v in action_counts.items() if 'interception' in k.lower())
    recoveries = sum(v for k, v in action_counts.items() if 'recovery' in k.lower())
    
    # Crear pitch
    pitch = Pitch(pitch_type='opta', pitch_color=PITCH_COLOR, line_color=LINE_COLOR, axis=False)
    fig, ax = pitch.draw(figsize=figsize)
    
    # KDE suave
    if len(player_data) >= 5:
        pitch.kdeplot(player_data['x'], player_data['y'], ax=ax, 
                     cmap=flamingo_cmap, fill=True, levels=15, alpha=0.7, thresh=0.05)
    else:
        pitch.scatter(player_data['x'], player_data['y'], ax=ax, 
                     s=200, c='#c03a1d', alpha=0.8, edgecolors='white', linewidth=1)
    
    # Título detallado
    title = f"{player_name} ({team_name})\nAcciones Defensivas (n={total_actions})"
    subtitle = f"Tackles: {tackles} | Intercepciones: {interceptions} | Recuperaciones: {recoveries}"
    
    ax.set_title(title, color=LINE_COLOR, fontsize=14, pad=20, fontweight='bold')
    ax.text(0.5, 0.00, subtitle, transform=ax.transAxes, ha='center', 
            color=LINE_COLOR, fontsize=10, alpha=0.8)
    
    fig.patch.set_facecolor(PITCH_COLOR)
    
    if save_image:
        safe_name = player_name.replace(" ", "_").replace(".", "")
        filename = f"acciones_defensivas_{safe_name}.png"
        filepath = SAVE_PATH / filename
        plt.savefig(filepath, dpi=300, bbox_inches='tight', 
                   facecolor=PITCH_COLOR, edgecolor='none')
        print(f"💾 Imagen guardada: {filepath}")
    
    if show_plot:
        plt.show()
    
    return fig, ax

def plot_top3_defensive_actions(actions_df, ranking_df, save_image=True, show_plot=True):
    """
    TOP 3 jugadores con más acciones defensivas.
    """
    top3 = ranking_df.head(3)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor(PITCH_COLOR)
    
    for i, (_, player) in enumerate(top3.iterrows()):
        player_id = player['player_id']
        player_name = player['player_name']
        team_name = player.get('team_name', 'Unknown')
        total_actions = player['total_actions']
        
        # Filtrar datos del jugador
        player_data = actions_df[actions_df['player_id'] == player_id]
        
        # Crear pitch
        pitch = Pitch(pitch_type='opta', pitch_color=PITCH_COLOR, 
                     line_color=LINE_COLOR, axis=False)
        pitch.draw(ax=axes[i])
        
        # KDE
        if not player_data.empty and len(player_data) >= 5:
            pitch.kdeplot(player_data['x'], player_data['y'], ax=axes[i], 
                         cmap=flamingo_cmap, fill=True, levels=12, alpha=0.7, thresh=0.05)
        elif not player_data.empty:
            pitch.scatter(player_data['x'], player_data['y'], ax=axes[i], 
                         s=150, c='#c03a1d', alpha=0.8, edgecolors='white', linewidth=1)
        
        # Título
        title = f"#{i+1} {player_name}\n({team_name})\n{total_actions} acciones defensivas"
        axes[i].set_title(title, color=LINE_COLOR, fontsize=11, pad=15, fontweight='bold')
    
    fig.suptitle("TOP 3 - Jugadores con Más Acciones Defensivas", 
                color=LINE_COLOR, fontsize=16, y=0.95, fontweight='bold')
    plt.tight_layout()
    
    if save_image:
        filepath = SAVE_PATH / "top3_acciones_defensivas.png"
        plt.savefig(filepath, dpi=300, bbox_inches='tight', 
                   facecolor=PITCH_COLOR, edgecolor='none')
        print(f"💾 TOP 3 guardado: {filepath}")
    
    if show_plot:
        plt.show()
    
    return fig

def search_defensive_players(ranking_df, search_term="", top_n=10):
    """Busca jugadores defensivos."""
    if search_term:
        mask = ranking_df['player_name'].str.lower().str.contains(search_term.lower(), na=False)
        results = ranking_df[mask].head(top_n)
    else:
        results = ranking_df.head(top_n)
    
    print(f"🛡️  Mejores jugadores defensivos:")
    for _, player in results.iterrows():
        actions = player.get('typeName', {})
        team = player.get('team_name', 'Unknown')
        print(f"  - {player['player_name']} ({team}) - {player['total_actions']} acciones")
        if isinstance(actions, dict):
            for action_type, count in actions.items():
                print(f"    → {action_type}: {count}")
    
    return results

# =====================================================================
# FUNCIÓN PRINCIPAL
# =====================================================================

def main_defensive():
    """Función principal para acciones defensivas."""
    try:
        print("🛡️  ANÁLISIS DE ACCIONES DEFENSIVAS")
        print("=" * 60)
        print("Incluye: Tackles, Intercepciones y Recuperaciones")
        
        if not BASE_DIR.exists():
            print(f"❌ El directorio base no existe: {BASE_DIR}")
            return None, None
        
        # Recopilar datos
        actions_df, ranking_df = collect_defensive_actions(BASE_DIR)
        
        # Estadísticas
        print(f"\n📈 Estadísticas:")
        print(f"   🛡️  Total acciones defensivas: {len(actions_df):,}")
        print(f"   👥 Jugadores únicos: {len(ranking_df):,}")
        
        # TOP 5
        print(f"\n🏆 TOP 5 Jugadores Defensivos:")
        for i, (_, player) in enumerate(ranking_df.head(5).iterrows()):
            team = player.get('team_name', 'Unknown')
            print(f"   {i+1}. {player['player_name']} ({team}) - {player['total_actions']} acciones")
        
        # Generar TOP 3
        print(f"\n🎨 Generando TOP 3 mapas defensivos...")
        plot_top3_defensive_actions(actions_df, ranking_df)
        
        print(f"\n✅ ¡Análisis defensivo completado!")
        return actions_df, ranking_df
        
    except Exception as e:
        print(f"❌ Error: {e}")
        raise

# =====================================================================
# EJECUCIÓN
# =====================================================================

if __name__ == "__main__":
    # Ejecutar análisis defensivo
    actions_df, ranking_df = main_defensive()
    
    # Ejemplo individual
    if actions_df is not None and not ranking_df.empty:
        top_defender = ranking_df.iloc[0]['player_name']
        print(f"\n🎯 Ejemplo - Mapa defensivo para: {top_defender}")
        plot_defensive_actions_heatmap(actions_df, top_defender)
        
        # Buscar jugadores
        print(f"\n🔍 Búsqueda de jugadores:")
        search_defensive_players(ranking_df, "", 5)

print("🛡️  ¡Notebook de acciones defensivas listo!")
print("📝 Funciones disponibles:")
print("  - plot_defensive_actions_heatmap(actions_df, 'Nombre Jugador')")
print("  - plot_top3_defensive_actions(actions_df, ranking_df)")
print("  - search_defensive_players(ranking_df, 'término búsqueda')")

In [ ]:
plot_defensive_actions_heatmap(actions_df, 'Pablo Fornals')

In [ ]:
# SOLUCIÓN CORREGIDA PARA MATCH_ID Y DETECCIÓN DE PORTEROS
# ========================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\matchcenter\MatchCenter\Competition\Season")

# =====================================================================
# FUNCIONES AUXILIARES CORREGIDAS
# =====================================================================

def find_match_directories(base_path, max_depth=5):
    """Encuentra directorios con archivos CSV - USA RUTA COMPLETA COMO ID."""
    match_dirs = []
    
    def explore_recursive(path, current_depth=0):
        if current_depth > max_depth or not path.is_dir():
            return
            
        try:
            csv_files = [f for f in path.iterdir() if f.is_file() and f.suffix.lower() == '.csv']
            
            if csv_files:
                csv_names = [f.name.lower() for f in csv_files]
                has_events = any('event' in name or 'defensive' in name for name in csv_names)
                has_players = any('player' in name for name in csv_names)
                
                if has_events or has_players or len(csv_files) > 3:
                    match_dirs.append(path)
            
            for subdir in path.iterdir():
                if subdir.is_dir():
                    explore_recursive(subdir, current_depth + 1)
                    
        except (PermissionError, OSError) as e:
            pass
    
    explore_recursive(base_path)
    return match_dirs

def create_unique_match_id(match_dir, players_df):
    """Crea un match_id único usando información del archivo players."""
    try:
        # Método 1: Si hay match_id en players.csv, usarlo
        if 'match_id' in players_df.columns and len(players_df) > 0:
            original_match_id = players_df['match_id'].iloc[0]
            if pd.notna(original_match_id):
                return str(original_match_id)
        
        # Método 2: Usar información de equipos
        if 'team_id' in players_df.columns and len(players_df) > 0:
            team_ids = sorted(players_df['team_id'].dropna().unique())
            if len(team_ids) >= 2:
                return f"match_{team_ids[0]}vs{team_ids[1]}"
        
        # Método 3: Usar nombres de equipos
        if 'team_name' in players_df.columns and len(players_df) > 0:
            team_names = sorted(players_df['team_name'].dropna().unique())
            if len(team_names) >= 2:
                safe_names = [name.replace(" ", "").replace(".", "") for name in team_names[:2]]
                return f"match_{safe_names[0]}vs{safe_names[1]}"
        
        # Método 4: Usar ruta relativa como último recurso
        relative_path = match_dir.relative_to(BASE_DIR)
        return str(relative_path).replace("\\", "_").replace("/", "_")
        
    except Exception as e:
        # Fallback: usar ruta completa
        return f"match_{hash(str(match_dir)) % 100000}"

def detect_goalkeepers_properly(players_df, events_df):
    """Detecta porteros usando SOLO la columna position del CSV players."""
    goalkeepers = set()
    
    # MÉTODO PRINCIPAL: Solo usar la columna 'position'
    if 'position' in players_df.columns:
        gk_mask = players_df['position'].str.upper() == 'GK'
        gk_players = players_df[gk_mask]['player_name'].unique()
        goalkeepers.update(gk_players)
        
        print(f"Porteros detectados por posición: {list(gk_players)}")
    else:
        print("No se encontró columna 'position' para detectar porteros")
    
    return goalkeepers

def filter_goalkeeper_events_properly(events_df, goalkeepers):
    """Filtra eventos de portero para que SOLO los tengan porteros reales."""
    if events_df.empty or not goalkeepers:
        return events_df
    
    # Eventos que SOLO pueden tener porteros
    GOALKEEPER_EVENTS = {
        71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
        103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 206
    }
    
    # Identificar eventos de portero en jugadores que NO son porteros
    non_gk_mask = (~events_df['player_name'].isin(goalkeepers)) & (events_df['typeValue'].isin(GOALKEEPER_EVENTS))
    
    if non_gk_mask.any():
        removed_count = non_gk_mask.sum()
        print(f"Removiendo {removed_count} eventos de portero de jugadores de campo")
        
        # Mostrar algunos ejemplos de lo que se está removiendo
        examples = events_df[non_gk_mask][['player_name', 'typeValue', 'typeName']].head(5)
        print("Ejemplos de eventos removidos:")
        for _, row in examples.iterrows():
            print(f"  - {row['player_name']}: {row['typeName']} (ID: {row['typeValue']})")
    
    # Filtrar eventos incorrectos
    filtered_df = events_df[~non_gk_mask].copy()
    
    return filtered_df

# =====================================================================
# FUNCIÓN PRINCIPAL CORREGIDA
# =====================================================================

def collect_player_data_fixed(base_dir):
    """Recopila datos con match_id único y detección correcta de porteros."""
    print("Recopilando datos con correcciones aplicadas...")
    
    match_dirs = find_match_directories(base_dir)
    if not match_dirs:
        raise RuntimeError(f"No se encontraron archivos CSV en {base_dir}")
    
    all_events = []
    processed_matches = 0
    match_id_tracker = {}  # Para verificar unicidad
    
    for i, match_dir in enumerate(match_dirs):
        print(f"Procesando {i+1}/{len(match_dirs)}: {match_dir}")
        
        # CORRECCIÓN: Buscar en la subcarpeta 'csv'
        csv_subdir = match_dir / 'csv'
        if csv_subdir.exists() and csv_subdir.is_dir():
            search_dir = csv_subdir
        else:
            search_dir = match_dir
        
        # Buscar archivos en el directorio correcto
        players_file = None
        events_file = None
        
        for file in search_dir.iterdir():
            if file.is_file() and file.suffix.lower() == '.csv':
                if file.name.lower() == 'players.csv':
                    players_file = file
                elif file.name.lower() == 'events.csv':
                    events_file = file
        
        if not players_file or not events_file:
            print(f"  SKIP: Archivos faltantes en {search_dir}")
            print(f"    Players: {'✓' if players_file else '✗'}")
            print(f"    Events: {'✓' if events_file else '✗'}")
            continue
        
        print(f"  Archivos encontrados:")
        print(f"    Players: {players_file.name}")
        print(f"    Events: {events_file.name}")
        
        # Leer archivos
        try:
            players_df = pd.read_csv(players_file)
            events_df = pd.read_csv(events_file)
            print(f"  Leídos: {len(players_df)} jugadores, {len(events_df)} eventos")
        except Exception as e:
            print(f"  ERROR leyendo archivos: {e}")
            continue
        
        # Normalizar columnas de players
        if 'player_id' not in players_df.columns:
            for col in players_df.columns:
                if any(pattern in col.lower() for pattern in ['player_id', 'playerid']):
                    players_df['player_id'] = pd.to_numeric(players_df[col], errors='coerce')
                    break
        
        if 'player_name' not in players_df.columns:
            for col in players_df.columns:
                if any(pattern in col.lower() for pattern in ['player_name', 'playername', 'name']):
                    players_df['player_name'] = players_df[col].astype(str)
                    break
        
        # Normalizar columnas de events
        if 'playerId' not in events_df.columns:
            for col in events_df.columns:
                if any(pattern in col.lower() for pattern in ['playerid', 'player_id']):
                    events_df['playerId'] = pd.to_numeric(events_df[col], errors='coerce')
                    break
        
        # Verificar columnas esenciales
        if 'player_id' not in players_df.columns or 'playerId' not in events_df.columns:
            print(f"  ERROR: Columnas esenciales faltantes")
            print(f"    Players cols: {list(players_df.columns)}")
            print(f"    Events cols: {list(events_df.columns)}")
            continue
        
        # CRÍTICO: Crear match_id único ANTES del merge
        unique_match_id = create_unique_match_id(match_dir, players_df)
        
        # Verificar que el match_id es realmente único
        if unique_match_id in match_id_tracker:
            # Hacer más único agregando índice
            unique_match_id = f"{unique_match_id}_part{i}"
            print(f"  WARNING: Match_ID duplicado, usando: {unique_match_id}")
        
        match_id_tracker[unique_match_id] = match_dir
        events_df['match_id'] = unique_match_id
        
        print(f"  Match_ID asignado: {unique_match_id}")
        
        # Detectar porteros usando SOLO la columna position
        goalkeepers = detect_goalkeepers_properly(players_df, events_df)
        
        # Merge con players
        events_with_players = events_df.merge(
            players_df[['player_id', 'player_name']], 
            left_on='playerId', 
            right_on='player_id', 
            how='left'
        )
        
        # Filtrar eventos válidos
        valid_events = events_with_players.dropna(subset=['player_name'])
        
        if valid_events.empty:
            print(f"  ERROR: Sin eventos válidos después del merge")
            continue
        
        # Aplicar filtro de porteros
        if goalkeepers:
            valid_events = filter_goalkeeper_events_properly(valid_events, goalkeepers)
        
        # Verificar que match_id se conservó
        if 'match_id' not in valid_events.columns:
            print(f"  ERROR: match_id perdido en el proceso")
            continue
        
        unique_matches_in_df = valid_events['match_id'].nunique()
        if unique_matches_in_df != 1:
            print(f"  WARNING: {unique_matches_in_df} match_ids únicos en DF (esperado: 1)")
        
        all_events.append(valid_events)
        processed_matches += 1
        
        print(f"  ÉXITO: {len(valid_events)} eventos, {valid_events['player_name'].nunique()} jugadores")
        print(f"         Match_ID verificado: {valid_events['match_id'].iloc[0]}")
    
    if not all_events:
        raise RuntimeError("No se procesaron eventos de ningún partido")
    
    print(f"\nConcatenando {len(all_events)} DataFrames...")
    
    # Verificar match_ids antes de concatenar
    print("Match_IDs antes de concatenar:")
    for i, df in enumerate(all_events[:5]):  # Solo mostrar primeros 5
        match_ids = df['match_id'].unique()
        print(f"  DF[{i}]: {match_ids}")
    
    combined_events = pd.concat(all_events, ignore_index=True)
    
    # Verificación final
    final_match_count = combined_events['match_id'].nunique()
    final_player_count = combined_events['player_name'].nunique()
    
    print(f"\nRESUMEN FINAL CORREGIDO:")
    print(f"  Partidos procesados: {processed_matches}/{len(match_dirs)}")
    print(f"  Partidos únicos en datos finales: {final_match_count}")
    print(f"  Total eventos: {len(combined_events)}")
    print(f"  Jugadores únicos: {final_player_count}")
    
    # Mostrar distribución de eventos por partido
    match_distribution = combined_events['match_id'].value_counts().head(10)
    print(f"\nDistribución de eventos por partido (top 10):")
    for match_id, count in match_distribution.items():
        print(f"  {match_id}: {count} eventos")
    
    return combined_events

# =====================================================================
# FUNCIÓN DE TESTING
# =====================================================================

def test_fixed_collection():
    """Prueba la colección de datos corregida."""
    try:
        print("PROBANDO COLECCIÓN DE DATOS CORREGIDA")
        print("=" * 50)
        
        combined_events = collect_player_data_fixed(BASE_DIR)
        
        # Verificaciones
        unique_matches = combined_events['match_id'].nunique()
        unique_players = combined_events['player_name'].nunique()
        
        print(f"\nVERIFICACIONES:")
        print(f"  ✅ Partidos únicos: {unique_matches} (debería ser > 1)")
        print(f"  ✅ Total eventos: {len(combined_events)}")
        print(f"  ✅ Jugadores únicos: {unique_players}")
        
        # Test de jugador específico en múltiples partidos
        top_players = combined_events['player_name'].value_counts().head(5)
        print(f"\nTOP 5 JUGADORES:")
        for player, count in top_players.items():
            matches_played = combined_events[combined_events['player_name'] == player]['match_id'].nunique()
            print(f"  {player}: {count} acciones, {matches_played} partidos")
        
        # Verificar porteros
        gk_events = [71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84]
        players_with_gk_events = combined_events[combined_events['typeValue'].isin(gk_events)]['player_name'].nunique()
        print(f"\nJugadores con eventos de portero: {players_with_gk_events} (debería ser ~6-10)")
        
        if unique_matches > 1:
            print(f"\n🎉 PROBLEMA RESUELTO: Se detectaron {unique_matches} partidos únicos")
        else:
            print(f"\n❌ Problema persiste: Solo {unique_matches} partido detectado")
        
        return combined_events
        
    except Exception as e:
        print(f"Error en test: {e}")
        return None

# =====================================================================
# EJECUCIÓN
# =====================================================================

if __name__ == "__main__":
    print("SISTEMA CORREGIDO CARGADO")
    print("Ejecuta: test_fixed_collection() para probar")
    
    # Ejecutar automáticamente
    result = test_fixed_collection()

In [ ]:
# =====================================================================
# SISTEMA SIMPLIFICADO DE ANÁLISIS DE JUGADORES PARA PIZZA CHARTS
# 
# Una sola visualización principal + análisis informativos en texto
# Objetivo: Identificar métricas óptimas para pizza charts de forma clara
# =====================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# -------------------------
# Configuración Visual
# -------------------------
BACKGROUND_COLOR = "#24282a"
TEXT_COLOR = "white"
ACCENT_COLOR = "#1A78CF"
SUCCESS_COLOR = "#00D4AA"
WARNING_COLOR = "#FFB800"
DANGER_COLOR = "#FF4757"

# Colores por nivel de percentil
PERCENTILE_COLORS = {
    'excelente': "#00D4AA",   # 90-100%
    'muy_bueno': "#1A78CF",   # 75-89%
    'bueno': "#FF9300",       # 60-74%
    'promedio': "#FFB800",    # 40-59%
    'bajo': "#FF4757"         # 0-39%
}

# Configuración principal
FIGSIZE = (14, 10)
FONTSIZE_TITLE = 16
FONTSIZE_SUBTITLE = 12
FONTSIZE_LABELS = 10
FONTSIZE_VALUES = 9

# Paths
SAVE_PATH = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\assets\viz\analysis")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

FBREF_CSV_PATH = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv")
TOP5_LEAGUES = {"Premier League", "La Liga", "Serie A", "Bundesliga", "Ligue 1"}

# =====================================================================
# Mapeo de Equipos
# =====================================================================
TEAM_NAME_MAPPING = {
    "Betis": "Real Betis", "Sociedad": "Real Sociedad", "Madrid": "Real Madrid",
    "Atlético": "Atlético Madrid", "Español": "Espanyol", "Celta": "Celta de Vigo",
    "Sevilla": "Sevilla FC", "Valencia": "Valencia CF", "Villarreal": "Villarreal CF",
    "Athletic": "Athletic Club", "Osasuna": "CA Osasuna", "Rayo": "Rayo Vallecano",
    "Getafe": "Getafe CF", "Leganés": "CD Leganés", "Las Palmas": "UD Las Palmas",
    "Girona": "Girona FC", "Mallorca": "RCD Mallorca", "Alavés": "Deportivo Alavés",
    "Valladolid": "Real Valladolid", "Manchester Utd": "Manchester United",
    "Manchester City": "Manchester City", "Tottenham": "Tottenham Hotspur",
    "Newcastle Utd": "Newcastle United", "West Ham": "West Ham United",
    "Brighton": "Brighton & Hove Albion", "Crystal Palace": "Crystal Palace",
    "Nott'ham Forest": "Nottingham Forest", "Inter": "Inter Milan",
    "AC Milan": "AC Milan", "Juventus": "Juventus", "Atalanta": "Atalanta BC",
    "Fiorentina": "ACF Fiorentina", "Lazio": "SS Lazio", "Roma": "AS Roma",
    "Napoli": "SSC Napoli", "Bayern Munich": "FC Bayern Munich",
    "Dortmund": "Borussia Dortmund", "RB Leipzig": "RB Leipzig",
    "Leverkusen": "Bayer Leverkusen", "Eint Frankfurt": "Eintracht Frankfurt",
    "Wolfsburg": "VfL Wolfsburg", "Gladbach": "Borussia Mönchengladbach",
    "Paris S-G": "Paris Saint-Germain", "Marseille": "Olympique Marseille",
    "Lyon": "Olympique Lyon", "Monaco": "AS Monaco", "Lille": "LOSC Lille",
    "Nice": "OGC Nice", "Rennes": "Stade Rennes", "Lens": "RC Lens"
}

def get_full_team_name(team_name: str) -> str:
    if pd.isna(team_name):
        return ""
    return TEAM_NAME_MAPPING.get(str(team_name).strip(), str(team_name).strip())

# =====================================================================
# Carga y Preparación de Datos
# =====================================================================

def load_data_for_analysis(csv_path: Path = FBREF_CSV_PATH) -> pd.DataFrame:
    """Carga datos con preparación completa."""
    df = pd.read_csv(csv_path)
    
    # Calcular 90s jugados
    df['nineties'] = df['partidos_completos_90'].fillna(0)
    mask_zero_90s = df['nineties'] <= 0
    df.loc[mask_zero_90s, 'nineties'] = df.loc[mask_zero_90s, 'minutos'] / 90.0
    df['nineties'] = df['nineties'].clip(lower=0.1)
    
    # Métricas principales para análisis
    key_metrics = [
        # Finalización
        'goles', 'goles_sin_penalti', 'xg', 'npxg', 'tiros', 'tiros_a_puerta',
        'conversion_goles', 'porc_tiros_a_puerta',
        
        # Creación
        'asistencias', 'xg_asistencias', 'pases_tercio_final', 'pases_area',
        'centros', 'centros_area', 'pases_al_hueco',
        
        # Pases
        'pases_progresivos', 'pases_completados', 'pases_largos_completados',
        'porc_precision_pase', 'porc_precision_pase_largo',
        
        # Posesión y regates
        'conducciones_progresivas', 'conducciones_tercio_final', 'conducciones_area',
        'regates_exitosos', 'porc_regates_exitosos', 'toques', 'toques_area_rival',
        
        # Defensa
        'entradas', 'intercepciones', 'entradas_mas_intercepciones', 'recuperaciones',
        'despejes', 'bloqueos', 'regates_parados', 'porc_regates_parados',
        
        # Duelos
        'duelos_aereos_ganados', 'porc_duelos_aereos_ganados',
        'faltas_cometidas', 'faltas_recibidas'
    ]
    
    # Calcular métricas por 90 para las principales
    base_metrics_for_per90 = [
        'goles', 'asistencias', 'xg', 'xg_asistencias', 'tiros', 'tiros_a_puerta',
        'pases_progresivos', 'conducciones_progresivas', 'regates_exitosos',
        'entradas', 'intercepciones', 'recuperaciones', 'despejes'
    ]
    
    for metric in base_metrics_for_per90:
        if metric in df.columns:
            per90_col = f"{metric}_por90"
            if per90_col not in df.columns:
                df[per90_col] = (df[metric].fillna(0) / df['nineties']).round(3)
    
    # Calcular métricas de porcentaje si no existen
    if 'conversion_goles' not in df.columns:
        df['conversion_goles'] = np.where(
            df['tiros'] > 0,
            (df['goles'] / df['tiros'] * 100).round(2),
            0
        )
    
    if 'porc_tiros_a_puerta' not in df.columns:
        df['porc_tiros_a_puerta'] = np.where(
            df['tiros'] > 0,
            (df['tiros_a_puerta'] / df['tiros'] * 100).round(2),
            0
        )
    
    # Información del jugador
    df['equipo'] = df['equipo'].apply(get_full_team_name)
    df['posicion_primaria'] = df['posicion'].astype(str).str.split(',').str[0].str.strip()
    
    return df

# =====================================================================
# Análisis Principal
# =====================================================================

def analyze_player_strengths(player_name: str, scope: str = "league", 
                           competition: str = None, min_percentile: int = 60,
                           min_minutes: int = 270) -> Dict:
    """
    Análisis completo de fortalezas del jugador.
    
    Returns:
        Dict con rankings, percentiles y recomendaciones
    """
    df = load_data_for_analysis()
    
    # Encontrar jugador
    player_mask = df["jugador"].str.lower() == player_name.lower()
    if not player_mask.any():
        return {"error": f"Jugador '{player_name}' no encontrado"}
    
    player_row = df[player_mask].iloc[0]
    player_position = player_row["posicion_primaria"]
    player_competition = str(player_row.get("competicion", ""))
    
    # Definir scope de comparación
    if scope.lower() == "league":
        if competition is None:
            competition = player_competition
        comparison_df = df[
            (df["competicion"] == competition) & 
            (df["posicion_primaria"] == player_position) &
            (df["minutos"] >= min_minutes)
        ].copy()
        scope_label = f"{competition} ({player_position})"
    elif scope.lower() == "top5":
        comparison_df = df[
            (df["competicion"].isin(TOP5_LEAGUES)) & 
            (df["posicion_primaria"] == player_position) &
            (df["minutos"] >= min_minutes)
        ].copy()
        scope_label = f"Top 5 Ligas ({player_position})"
    
    # Obtener métricas relevantes
    exclude_cols = [
        'edad', 'año_nacimiento', 'partidos_jugados', 'partidos_titular', 'minutos',
        'partidos_completos_90', 'nineties', 'tarjetas_amarillas', 'tarjetas_rojas',
        'penales_anotados', 'penales_intentados', 'autogoles', 'errores'
    ]
    
    numeric_cols = comparison_df.select_dtypes(include=[np.number]).columns.tolist()
    analysis_cols = [col for col in numeric_cols if col not in exclude_cols]
    
    # Calcular percentiles y rankings
    results = {
        'player_info': {
            'name': player_name,
            'position': player_position,
            'team': player_row.get('equipo', ''),
            'competition': player_competition,
            'scope': scope_label
        },
        'top_rankings': [],
        'high_percentiles': [],
        'por90_better': [],
        'recommendations_by_category': {}
    }
    
    for metric in analysis_cols:
        metric_data = comparison_df[metric].dropna()
        if len(metric_data) < 5:
            continue
        
        player_value = float(player_row.get(metric, 0))
        
        # Calcular ranking (posición absoluta)
        ascending = metric in ['faltas_cometidas', 'perdidas', 'malos_controles']
        ranking = int(metric_data.rank(ascending=ascending, method='min')[player_mask.loc[comparison_df.index]].iloc[0])
        
        # Calcular percentil
        if ascending:
            percentile = (metric_data > player_value).mean() * 100
        else:
            percentile = (metric_data < player_value).mean() * 100
        percentile = round(max(0, min(100, percentile)), 1)
        
        # Almacenar si está en top 5
        if ranking <= 5:
            results['top_rankings'].append({
                'metrica': metric,
                'ranking': ranking,
                'valor': round(player_value, 3),
                'percentil': percentile,
                'categoria': categorize_metric(metric)
            })
        
        # Almacenar si percentil es alto
        if percentile >= min_percentile:
            results['high_percentiles'].append({
                'metrica': metric,
                'percentil': percentile,
                'valor': round(player_value, 3),
                'categoria': categorize_metric(metric),
                'nivel': classify_percentile(percentile)
            })
    
    # Comparar métricas por90 vs totales
    base_metrics = ['goles', 'asistencias', 'xg', 'xg_asistencias', 'tiros', 
                   'pases_progresivos', 'regates_exitosos', 'entradas', 'intercepciones']
    
    for metric in base_metrics:
        total_col = metric
        per90_col = f"{metric}_por90"
        
        if total_col in comparison_df.columns and per90_col in comparison_df.columns:
            total_percentile = calculate_percentile(comparison_df, player_row, total_col)
            per90_percentile = calculate_percentile(comparison_df, player_row, per90_col)
            
            if total_percentile is not None and per90_percentile is not None:
                diff = per90_percentile - total_percentile
                if diff >= 10:  # Por90 significativamente mejor
                    results['por90_better'].append({
                        'metrica': metric,
                        'percentil_total': total_percentile,
                        'percentil_por90': per90_percentile,
                        'diferencia': diff
                    })
    
    # Agrupar recomendaciones por categoría
    all_strong_metrics = set()
    
    # De rankings top 3
    top_3_metrics = [r['metrica'] for r in results['top_rankings'] if r['ranking'] <= 3]
    all_strong_metrics.update(top_3_metrics)
    
    # De percentiles altos (>=75)
    high_perc_metrics = [r['metrica'] for r in results['high_percentiles'] if r['percentil'] >= 75]
    all_strong_metrics.update(high_perc_metrics)
    
    # Métricas por90 recomendadas
    recommended_por90 = [f"{r['metrica']}_por90" for r in results['por90_better']]
    all_strong_metrics.update(recommended_por90)
    
    # Agrupar por categorías
    for metric in all_strong_metrics:
        category = categorize_metric(metric)
        if category not in results['recommendations_by_category']:
            results['recommendations_by_category'][category] = []
        results['recommendations_by_category'][category].append(metric)
    
    return results

def calculate_percentile(df: pd.DataFrame, player_row: pd.Series, metric: str) -> float:
    """Calcula percentil de una métrica específica."""
    metric_data = df[metric].dropna()
    if len(metric_data) < 5:
        return None
    
    player_value = float(player_row.get(metric, 0))
    ascending = metric not in ['faltas_cometidas', 'perdidas', 'malos_controles']
    
    if ascending:
        percentile = (metric_data < player_value).mean() * 100
    else:
        percentile = (metric_data > player_value).mean() * 100
    
    return round(max(0, min(100, percentile)), 1)

def classify_percentile(percentile: float) -> str:
    """Clasifica percentil en niveles."""
    if percentile >= 90:
        return 'excelente'
    elif percentile >= 75:
        return 'muy_bueno'
    elif percentile >= 60:
        return 'bueno'
    elif percentile >= 40:
        return 'promedio'
    else:
        return 'bajo'

def categorize_metric(metric: str) -> str:
    """Categoriza métrica por tipo."""
    if any(x in metric.lower() for x in ['gol', 'tiro', 'npxg', 'xg', 'conversion']):
        return 'Finalización'
    elif any(x in metric.lower() for x in ['asist', 'pase', 'xa', 'centro', 'hueco']):
        return 'Creación'
    elif any(x in metric.lower() for x in ['entrada', 'intercep', 'recup', 'despej', 'bloq']):
        return 'Defensa'
    elif any(x in metric.lower() for x in ['regate', 'conducc', 'toque']):
        return 'Posesión'
    elif any(x in metric.lower() for x in ['duelo', 'aerial', 'falta']):
        return 'Duelos'
    elif any(x in metric.lower() for x in ['porc_', 'precision', '%', 'conversion']):
        return 'Eficiencia'
    else:
        return 'Otros'

def format_metric_display(metric: str) -> str:
    """Formatea nombre de métrica para display."""
    name_mapping = {
        'goles_por90': 'Goles/90', 'asistencias_por90': 'Asistencias/90',
        'xg_por90': 'xG/90', 'xg_asistencias_por90': 'xA/90', 'npxg_por90': 'npxG/90',
        'tiros_por90': 'Tiros/90', 'tiros_a_puerta_por90': 'Tiros portería/90',
        'pases_progresivos_por90': 'Pases progresivos/90',
        'conducciones_progresivas_por90': 'Conducciones progresivas/90',
        'regates_exitosos_por90': 'Regates exitosos/90',
        'entradas_por90': 'Entradas/90', 'intercepciones_por90': 'Intercepciones/90',
        'porc_precision_pase': '% Precisión pases', 'conversion_goles': '% Conversión goles',
        'porc_duelos_aereos_ganados': '% Duelos aéreos', 'porc_tiros_a_puerta': '% Tiros portería',
        'entradas_mas_intercepciones': 'Entradas + Intercepciones',
        'conducciones_tercio_final': 'Conducciones tercio final'
    }
    
    return name_mapping.get(metric, metric.replace('_', ' ').title())

# =====================================================================
# Visualización Principal
# =====================================================================

def create_player_strengths_chart(player_name: str, scope: str = "league", 
                                min_percentile: int = 60, save_image: bool = True):
    """Visualización única con mejores percentiles del jugador."""
    
    # Obtener análisis
    analysis = analyze_player_strengths(player_name, scope, min_percentile=min_percentile)
    
    if 'error' in analysis:
        print(f"❌ {analysis['error']}")
        return None
    
    percentiles_data = analysis['high_percentiles']
    
    if not percentiles_data:
        print(f"❌ {player_name} no tiene percentiles ≥{min_percentile}%")
        return None
    
    # Ordenar por percentil descendente y tomar top 15
    percentiles_data = sorted(percentiles_data, key=lambda x: x['percentil'], reverse=True)[:15]
    
    # Crear figura
    fig, ax = plt.subplots(figsize=FIGSIZE, facecolor=BACKGROUND_COLOR)
    ax.set_facecolor(BACKGROUND_COLOR)
    
    # Datos para el gráfico
    metrics = [d['metrica'] for d in percentiles_data]
    percentiles = [d['percentil'] for d in percentiles_data]
    values = [d['valor'] for d in percentiles_data]
    colors = [PERCENTILE_COLORS[d['nivel']] for d in percentiles_data]
    
    # Crear barras horizontales
    y_pos = np.arange(len(metrics)) * 1.2
    bars = ax.barh(y_pos, percentiles, color=colors, alpha=0.8, height=0.7)
    
    # Añadir etiquetas de valores
    for i, (percentile, value) in enumerate(zip(percentiles, values)):
        # Percentil en la barra
        ax.text(percentile + 1, y_pos[i], f"{percentile:.0f}%", 
                va='center', ha='left', fontweight='bold', 
                color=TEXT_COLOR, fontsize=FONTSIZE_VALUES)
        
        # Valor de la métrica a la izquierda
        ax.text(-2, y_pos[i], f"{value:.2f}", va='center', ha='right',
                color=TEXT_COLOR, fontsize=FONTSIZE_VALUES, fontweight='bold')
    
    # Configurar ejes
    ax.set_yticks(y_pos)
    ax.set_yticklabels([format_metric_display(m) for m in metrics], 
                       color=TEXT_COLOR, fontsize=FONTSIZE_LABELS)
    ax.tick_params(axis='y', pad=55)
    ax.set_xlim(0, 105)
    ax.set_xlabel('Percentil vs Compañeros de Posición', color=TEXT_COLOR, fontsize=FONTSIZE_LABELS)
    
    # Líneas de referencia
    for line_val, alpha, label in [(60, 0.2, '60%'), (75, 0.3, '75%'), (90, 0.5, '90%')]:
        ax.axvline(x=line_val, color=TEXT_COLOR, alpha=alpha, linestyle='--', linewidth=1)
        ax.text(line_val, len(percentiles_data), label, 
                ha='center', va='bottom', color=TEXT_COLOR, fontsize=8)
    
    # Títulos
    player_info = analysis['player_info']
    ax.set_title(f"Fortalezas: {player_info['name']}", 
                color=TEXT_COLOR, fontsize=FONTSIZE_TITLE, fontweight='bold', pad=20)
    
    subtitle = f"{player_info['team']} | {player_info['scope']} | Percentil ≥{min_percentile}%"
    ax.text(0.5, 0.97, subtitle, transform=ax.transAxes, ha='center', 
            color=TEXT_COLOR, fontsize=FONTSIZE_SUBTITLE-2)
    
    # Leyenda de niveles
    legend_elements = []
    legend_labels = []
    for nivel, color in PERCENTILE_COLORS.items():
        if any(d['nivel'] == nivel for d in percentiles_data):
            legend_elements.append(plt.Rectangle((0,0),1,1, facecolor=color, alpha=0.8))
            nivel_names = {
                'excelente': 'Excelente (90-100%)',
                'muy_bueno': 'Muy Bueno (75-89%)',
                'bueno': 'Bueno (60-74%)',
                'promedio': 'Promedio (40-59%)',
                'bajo': 'Bajo (0-39%)'
            }
            legend_labels.append(nivel_names[nivel])
    
    ax.legend(legend_elements, legend_labels, loc='lower right', frameon=False,
              fontsize=FONTSIZE_LABELS-2, title='Nivel de Percentil')
    
    # Estilo
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=TEXT_COLOR)
    ax.grid(axis='x', alpha=0.3, color=TEXT_COLOR)
    
    plt.tight_layout()
    
    # Guardar imagen
    if save_image:
        safe_name = player_name.replace(" ", "_")
        filepath = SAVE_PATH / f"strengths_{safe_name}_{scope}_min{min_percentile}.png"
        plt.savefig(filepath, dpi=300, bbox_inches="tight", 
                   facecolor=BACKGROUND_COLOR)
        print(f"✅ Gráfico guardado: {filepath}")
    
    plt.show()
    return fig, analysis

# =====================================================================
# Función Principal de Análisis
# =====================================================================

def analyze_player_for_pizza_chart(player_name: str, scope: str = "league", 
                                 min_percentile: int = 60):
    """
    Función principal que genera el análisis completo del jugador.
    
    Incluye:
    1. Visualización de fortalezas
    2. Análisis textual detallado
    3. Recomendaciones para pizza chart
    """
    print(f"🔍 ANÁLISIS PARA PIZZA CHART: {player_name}")
    print("=" * 60)
    
    # Generar visualización y análisis
    result = create_player_strengths_chart(player_name, scope, min_percentile, save_image=True)
    
    if result is None:
        return None
    
    fig, analysis = result
    
    # Mostrar análisis textual
    print(f"\n📊 RESUMEN DE FORTALEZAS:")
    print("-" * 40)
    
    # Top Rankings
    top_rankings = analysis['top_rankings']
    if top_rankings:
        print(f"\n🏆 RANKINGS TOP 5 ({len(top_rankings)} métricas):")
        for rank_data in sorted(top_rankings, key=lambda x: x['ranking'])[:6]:
            print(f"   #{rank_data['ranking']} - {format_metric_display(rank_data['metrica']):30} | {rank_data['valor']:6.2f} ({rank_data['percentil']:3.0f}%)")
    
    # Mejores Percentiles
    high_percentiles = analysis['high_percentiles']
    if high_percentiles:
        print(f"\n📈 MEJORES PERCENTILES (≥{min_percentile}%, {len(high_percentiles)} métricas):")
        for perc_data in sorted(high_percentiles, key=lambda x: x['percentil'], reverse=True)[:8]:
            nivel_symbol = {"excelente": "🟢", "muy_bueno": "🔵", "bueno": "🟡", "promedio": "🟠", "bajo": "🔴"}
            symbol = nivel_symbol.get(perc_data['nivel'], "⚪")
            print(f"   {symbol} {perc_data['percentil']:3.0f}% - {format_metric_display(perc_data['metrica']):30} | {perc_data['valor']:6.2f}")
    
    # Mejor Por90
    por90_better = analysis['por90_better']
    if por90_better:
        print(f"\n⚡ MEJOR EN POR/90 ({len(por90_better)} métricas):")
        for por90_data in sorted(por90_better, key=lambda x: x['diferencia'], reverse=True):
            print(f"   📊 {format_metric_display(por90_data['metrica'])}_por90 | Diferencia: +{por90_data['diferencia']:2.0f}% vs totales")
    
    # Recomendaciones por categoría
    recommendations = analysis['recommendations_by_category']
    if recommendations:
        print(f"\n🎯 RECOMENDACIONES PARA PIZZA CHART:")
        print("-" * 50)
        
        category_symbols = {
            'Finalización': '⚽', 'Creación': '🎯', 'Defensa': '🛡️',
            'Posesión': '⚡', 'Duelos': '💪', 'Eficiencia': '📊'
        }
        
        for category, metrics in recommendations.items():
            symbol = category_symbols.get(category, '📈')
            print(f"\n   {symbol} {category} ({len(metrics)} métricas):")
            for metric in metrics[:4]:  # Top 4 por categoría
                display_name = format_metric_display(metric)
                print(f"     ✅ {display_name}")
    
    print(f"\n💡 SIGUIENTE PASO: Usar las métricas recomendadas en el pizza chart")
    print(f"   Las métricas con percentiles ≥75% son ideales para mostrar fortalezas")
    
    return analysis

# =====================================================================
# Ejemplos de Uso
# =====================================================================

if __name__ == "__main__":
    # Análisis individual
    analyze_player_for_pizza_chart("Pedri", scope="top5", min_percentile=70)
    
    # Otros ejemplos (descomenta para usar)
    # analyze_player_for_pizza_chart("Pedri", scope="league", min_percentile=70)
    # analyze_player_for_pizza_chart("Luka Modrić", scope="top5", min_percentile=65)

def analyze_multiple_players(players: list, scope: str = "league", min_percentile: int = 60):
    """Analiza múltiples jugadores de forma consecutiva."""
    print(f"🔍 ANALIZANDO {len(players)} JUGADORES")
    print("=" * 60)
    
    all_results = {}
    
    for i, player in enumerate(players, 1):
        print(f"\n[{i}/{len(players)}] 📊 {player}")
        print("-" * 40)
        
        try:
            result = analyze_player_for_pizza_chart(player, scope, min_percentile)
            all_results[player] = result
            print(f"✅ {player} - Completado")
            
        except Exception as e:
            print(f"❌ Error con {player}: {e}")
            all_results[player] = None
        
        if i < len(players):
            print("\n" + "="*40 + "\n")
    
    print(f"\n🎉 ANÁLISIS MÚLTIPLE COMPLETADO")
    print(f"   ✅ Exitosos: {len([r for r in all_results.values() if r])}")
    print(f"   ❌ Errores: {len([r for r in all_results.values() if not r])}")
    
    return all_results

# =====================================================================
# Función de Comparación Rápida
# =====================================================================

def quick_compare_players(players: list, scope: str = "league", top_n: int = 5):
    """Comparación rápida entre jugadores mostrando sus top métricas."""
    print(f"🔥 COMPARACIÓN RÁPIDA - TOP {top_n} FORTALEZAS")
    print("=" * 70)
    
    df = load_data_for_analysis()
    comparison_results = {}
    
    for player_name in players:
        # Obtener análisis básico
        analysis = analyze_player_strengths(player_name, scope, min_percentile=70)
        
        if 'error' not in analysis:
            # Obtener top N percentiles
            top_percentiles = sorted(analysis['high_percentiles'], 
                                   key=lambda x: x['percentil'], reverse=True)[:top_n]
            comparison_results[player_name] = top_percentiles
    
    # Mostrar comparación
    for player_name, percentiles in comparison_results.items():
        print(f"\n🎯 {player_name}:")
        if percentiles:
            for i, data in enumerate(percentiles, 1):
                print(f"   {i}. {format_metric_display(data['metrica']):25} | {data['percentil']:3.0f}% ({data['valor']:.2f})")
        else:
            print("   Sin fortalezas destacadas")
    
    return comparison_results

# =====================================================================
# Análisis por Posición
# =====================================================================

def analyze_position_benchmarks(position: str, competition: str = None, 
                               scope: str = "league", min_minutes: int = 270):
    """Analiza benchmarks de una posición específica."""
    df = load_data_for_analysis()
    
    # Filtrar por posición y competición
    if scope.lower() == "league" and competition:
        position_df = df[
            (df["posicion_primaria"] == position) & 
            (df["competicion"] == competition) &
            (df["minutos"] >= min_minutes)
        ]
        scope_label = f"{competition} - {position}"
    elif scope.lower() == "top5":
        position_df = df[
            (df["posicion_primaria"] == position) & 
            (df["competicion"].isin(TOP5_LEAGUES)) &
            (df["minutos"] >= min_minutes)
        ]
        scope_label = f"Top 5 Ligas - {position}"
    
    if len(position_df) < 10:
        print(f"❌ Pocos jugadores encontrados para {position} en {scope_label}")
        return None
    
    print(f"📊 BENCHMARKS PARA {scope_label}")
    print(f"Muestra: {len(position_df)} jugadores")
    print("=" * 50)
    
    # Métricas clave por posición
    position_metrics = {
        'DF': ['despejes', 'intercepciones', 'entradas', 'duelos_aereos_ganados', 'recuperaciones'],
        'LB': ['pases_progresivos', 'centros', 'entradas', 'intercepciones', 'recuperaciones'],
        'RB': ['pases_progresivos', 'centros', 'entradas', 'intercepciones', 'recuperaciones'],
        'DM': ['pases_progresivos', 'intercepciones', 'entradas', 'recuperaciones', 'pases_completados'],
        'CM': ['pases_progresivos', 'pases_completados', 'conducciones_progresivas', 'intercepciones'],
        'AM': ['xg_asistencias', 'pases_tercio_final', 'regates_exitosos', 'conducciones_progresivas'],
        'LM': ['centros', 'regates_exitosos', 'conducciones_progresivas', 'pases_tercio_final'],
        'RM': ['centros', 'regates_exitosos', 'conducciones_progresivas', 'pases_tercio_final'],
        'LW': ['regates_exitosos', 'conducciones_progresivas', 'centros', 'tiros'],
        'RW': ['regates_exitosos', 'conducciones_progresivas', 'centros', 'tiros'],
        'FW': ['goles', 'xg', 'tiros_a_puerta', 'regates_exitosos']
    }
    
    key_metrics = position_metrics.get(position, ['goles', 'asistencias', 'pases_completados'])
    
    # Calcular percentiles de referencia
    benchmarks = {}
    for metric in key_metrics:
        if metric in position_df.columns:
            metric_data = position_df[metric].dropna()
            if len(metric_data) > 5:
                benchmarks[metric] = {
                    'p50': round(metric_data.quantile(0.5), 2),
                    'p75': round(metric_data.quantile(0.75), 2),
                    'p90': round(metric_data.quantile(0.9), 2),
                    'max': round(metric_data.max(), 2)
                }
    
    # Mostrar benchmarks
    print(f"\n🎯 BENCHMARKS CLAVE:")
    print("-" * 30)
    for metric, values in benchmarks.items():
        print(f"{format_metric_display(metric):25}")
        print(f"   P50: {values['p50']:6.2f} | P75: {values['p75']:6.2f} | P90: {values['p90']:6.2f} | Max: {values['max']:6.2f}")
    
    return benchmarks

# =====================================================================
# Función Todo-en-Uno
# =====================================================================

def pizza_analysis_suite(player_name: str, scope: str = "league", 
                        min_percentile: int = 60, competition: str = None):
    """
    Suite completa de análisis para pizza charts.
    
    Incluye:
    - Análisis principal del jugador
    - Benchmarks de su posición
    - Recomendaciones específicas
    """
    
    print(f"🍕 PIZZA CHART ANALYSIS SUITE")
    print(f"Jugador: {player_name} | Scope: {scope}")
    print("=" * 70)
    
    # 1. Análisis principal
    print("PASO 1: Análisis de fortalezas del jugador")
    print("-" * 45)
    
    analysis = analyze_player_for_pizza_chart(player_name, scope, min_percentile)
    
    if analysis is None:
        return None
    
    # 2. Benchmarks de posición
    print(f"\n\nPASO 2: Benchmarks de posición")
    print("-" * 35)
    
    player_position = analysis['player_info']['position']
    player_competition = competition or analysis['player_info']['competition']
    
    benchmarks = analyze_position_benchmarks(
        player_position, player_competition, scope
    )
    
    # 3. Recomendaciones finales
    print(f"\n\n🎯 RECOMENDACIONES FINALES PARA PIZZA CHART")
    print("=" * 55)
    
    recommendations = analysis['recommendations_by_category']
    
    # Seleccionar mejores métricas (8-12 para pizza chart)
    final_pizza_metrics = []
    
    # Priorizar métricas con percentil ≥75
    high_perc_metrics = [m for m in analysis['high_percentiles'] 
                        if m['percentil'] >= 75]
    
    # Agrupar por categoría y seleccionar las mejores
    categories_selected = {}
    for metric_data in sorted(high_perc_metrics, key=lambda x: x['percentil'], reverse=True):
        category = metric_data['categoria']
        if category not in categories_selected:
            categories_selected[category] = []
        if len(categories_selected[category]) < 3:  # Max 3 por categoría
            categories_selected[category].append(metric_data['metrica'])
            final_pizza_metrics.append({
                'metrica': metric_data['metrica'],
                'percentil': metric_data['percentil'],
                'categoria': category,
                'valor': metric_data['valor']
            })
    
    # Mostrar selección final
    print(f"\n✅ MÉTRICAS RECOMENDADAS PARA PIZZA CHART ({len(final_pizza_metrics)} seleccionadas):")
    print("-" * 60)
    
    for i, metric_data in enumerate(final_pizza_metrics[:12], 1):  # Top 12
        category_symbols = {
            'Finalización': '⚽', 'Creación': '🎯', 'Defensa': '🛡️',
            'Posesión': '⚡', 'Duelos': '💪', 'Eficiencia': '📊', 'Otros': '📈'
        }
        symbol = category_symbols.get(metric_data['categoria'], '📈')
        
        print(f"{i:2d}. {symbol} {format_metric_display(metric_data['metrica']):30} | "
              f"{metric_data['percentil']:3.0f}% | {metric_data['valor']:6.2f}")
    
    print(f"\n💡 TIPS PARA EL PIZZA CHART:")
    print("• Usar métricas con percentil ≥70% para mostrar fortalezas")
    print("• Incluir 2-3 métricas por categoría principal")
    print("• Considerar métricas por/90 cuando sean significativamente mejores")
    print("• Balancear entre métricas ofensivas y defensivas según posición")
    
    return {
        'analysis': analysis,
        'benchmarks': benchmarks,
        'final_metrics': final_pizza_metrics
    }

In [ ]:
# =====================================================================
# PIZZA FBREF v4 FINAL (ES) - Métricas robustas + Layout perfecto
# Soluciones implementadas:
# - Radar más pequeño para evitar solapamiento con título
# - Métricas con fallbacks robustos, sin valores 0
# - Mapeo de nombres de equipos completos
# - Sistema de métricas alternativas cuando faltan datos
# - Verificación exhaustiva de disponibilidad de datos
# =====================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from mplsoccer import PyPizza

# -------------------------
# Configuración Visual Optimizada
# -------------------------
BACKGROUND_COLOR = "#24282a"
TEXT_COLOR = "white"

# Paleta de colores
OFFENSIVE_COLOR = "#1A78CF"   # Ataque (Azul)
CREATIVE_COLOR  = "#FF9300"   # Creación/Posesión (Naranja) 
DEFENSIVE_COLOR = "#D70232"   # Defensa (Rojo)
GRID_LINE_COLOR = "#31363a"

# Tamaños ajustados para radar más pequeño
FIGSIZE_SINGLE   = (8, 9.5)        # Más alto para mejor espaciado
FIGSIZE_COMPARE  = (18, 10)
PARAM_FONTSIZE   = 7.5            # Más pequeño
VALUE_FONTSIZE   = 8
TITLE_FONTSIZE   = 16
SUBTITLE_FONTSIZE= 11
GROUPLABEL_FONTSIZE = 11

# Configuración del radar - MÁS PEQUEÑO
INNER_CIRCLE_SIZE = 28             # Mayor círculo interior
STRAIGHT_LINE_LW = 0.8             # Líneas más finas

# Paths
SAVE_PATH = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\assets\viz\pizza_charts")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

FBREF_CSV_PATH = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv")

TOP5 = {"Premier League", "La Liga", "Serie A", "Bundesliga", "Ligue 1"}

# =====================================================================
# Mapeo de Nombres de Equipos
# =====================================================================

TEAM_NAME_MAPPING = {
    # La Liga
    "Betis": "Real Betis",
    "Sociedad": "Real Sociedad", 
    "Madrid": "Real Madrid",
    "Atlético": "Atlético Madrid",
    "Español": "Espanyol",
    "Celta": "Celta de Vigo",
    "Sevilla": "Sevilla FC",
    "Valencia": "Valencia CF",
    "Villarreal": "Villarreal CF",
    "Athletic": "Athletic Club",
    "Osasuna": "CA Osasuna",
    "Rayo": "Rayo Vallecano",
    "Getafe": "Getafe CF",
    "Leganés": "CD Leganés",
    "Las Palmas": "UD Las Palmas",
    "Girona": "Girona FC",
    "Mallorca": "RCD Mallorca",
    "Alavés": "Deportivo Alavés",
    "Valladolid": "Real Valladolid",
    
    # Premier League
    "Manchester Utd": "Manchester United",
    "Manchester City": "Manchester City",
    "Tottenham": "Tottenham Hotspur",
    "Newcastle Utd": "Newcastle United",
    "West Ham": "West Ham United",
    "Brighton": "Brighton & Hove Albion",
    "Crystal Palace": "Crystal Palace",
    "Nott'ham Forest": "Nottingham Forest",
    
    # Serie A
    "Inter": "Inter Milan",
    "AC Milan": "AC Milan",
    "Juventus": "Juventus",
    "Atalanta": "Atalanta BC",
    "Fiorentina": "ACF Fiorentina",
    "Lazio": "SS Lazio",
    "Roma": "AS Roma",
    "Napoli": "SSC Napoli",
    "Bologna": "Bologna FC",
    "Torino": "Torino FC",
    "Udinese": "Udinese Calcio",
    
    # Bundesliga
    "Bayern Munich": "FC Bayern Munich",
    "Dortmund": "Borussia Dortmund",
    "RB Leipzig": "RB Leipzig",
    "Leverkusen": "Bayer Leverkusen",
    "Eint Frankfurt": "Eintracht Frankfurt",
    "Wolfsburg": "VfL Wolfsburg",
    "Gladbach": "Borussia Mönchengladbach",
    
    # Ligue 1
    "Paris S-G": "Paris Saint-Germain",
    "Marseille": "Olympique Marseille",
    "Lyon": "Olympique Lyon",
    "Monaco": "AS Monaco",
    "Lille": "LOSC Lille",
    "Nice": "OGC Nice",
    "Rennes": "Stade Rennes",
    "Lens": "RC Lens"
}

def get_full_team_name(team_name: str) -> str:
    """Convierte nombres cortos a nombres completos de equipos."""
    if pd.isna(team_name):
        return ""
    return TEAM_NAME_MAPPING.get(str(team_name).strip(), str(team_name).strip())

# =====================================================================
# Carga y Preparación Robusta de Datos  
# =====================================================================

def load_fbref_players(csv_path: Path = FBREF_CSV_PATH) -> pd.DataFrame:
    """Carga datos de FBRef con cálculos robustos de métricas."""
    df = pd.read_csv(csv_path)
    
    # Calcular 90s jugados de forma robusta
    df['nineties'] = df['partidos_completos_90'].fillna(0)
    mask_zero_90s = df['nineties'] <= 0
    df.loc[mask_zero_90s, 'nineties'] = df.loc[mask_zero_90s, 'minutos'] / 90.0
    df['nineties'] = df['nineties'].clip(lower=0.1)  # Mínimo 0.1 para evitar división por 0
    
    # Métricas base verificadas (las que SÍ existen en FBRef)
    verified_base_metrics = [
        'tiros_asistidos', 'pases_area', 'pases_tercio_final', 'regates_exitosos',
        'pases_progresivos', 'conducciones_progresivas', 'pases_al_hueco',
        'entradas_mas_intercepciones', 'pases_bloqueados', 'faltas_cometidas',
        'recuperaciones', 'entradas', 'intercepciones', 'bloqueos', 'despejes',
        'centros', 'conducciones_tercio_final', 'conducciones_area', 'toques',
        'perdidas', 'duelos_aereos_ganados', 'duelos_aereos_perdidos',
        'goles_sin_penalti', 'tiros', 'toques_area_rival', 'cambios_de_juego', 'pases_largos_completados'
    ]
    
    # Calcular métricas por 90 con verificaciones
    for metric in verified_base_metrics:
        if metric in df.columns:
            per90_col = f"{metric}_por90"
            if per90_col not in df.columns:
                df[per90_col] = (df[metric].fillna(0) / df['nineties']).round(2)
    
    # Métricas derivadas robustas
    
    # % duelos aéreos ganados con fallback
    if 'porc_duelos_aereos_ganados' not in df.columns:
        total_aereos = df.get('duelos_aereos_ganados', 0) + df.get('duelos_aereos_perdidos', 0)
        df['porc_duelos_aereos_ganados'] = np.where(
            total_aereos > 0,
            (df.get('duelos_aereos_ganados', 0) / total_aereos * 100).round(1),
            0.0
        )
    
    # % tiros a portería con verificación
    if 'porcentaje_tiros_a_porteria' not in df.columns:
        if 'porc_tiros_a_puerta' in df.columns:
            df['porcentaje_tiros_a_porteria'] = df['porc_tiros_a_puerta']
        else:
            # Calcular desde tiros base
            tiros_total = df.get('tiros', 1)  # Evitar división por 0
            df['porcentaje_tiros_a_porteria'] = np.where(
                tiros_total > 0,
                (df.get('tiros_a_puerta', 0) / tiros_total * 100).round(1),
                0.0
            )
    
    # Toques por pérdida (métrica de retención) - MUY IMPORTANTE
    if 'toques_por90' in df.columns and 'perdidas_por90' in df.columns:
        df['toques_por_perdida_por90'] = np.where(
            df['perdidas'] > 0,
            (df['toques'] / df['perdidas']).round(1),
            df['toques'].fillna(0)  # Si no hay pérdidas, usar toques directamente
        )
    else:
        # Fallback si no existen estas columnas
        df['toques_por_perdida'] = 10.0  # Valor promedio razonable
    
    # Verificar/renombrar columnas existentes
    if 'porc_precision_pase' not in df.columns and 'porcentaje_pases_completados' in df.columns:
        df['porc_precision_pase'] = df['porcentaje_pases_completados']
    elif 'porc_precision_pase' not in df.columns:
        df['porc_precision_pase'] = 85.0  # Fallback promedio
    
    if 'porc_precision_pase_largo' not in df.columns and 'porcentaje_pases_largos_completados' in df.columns:
        df['porc_precision_pase_largo'] = df['porcentaje_pases_largos_completados'] 
    elif 'porc_precision_pase_largo' not in df.columns:
        df['porc_precision_pase_largo'] = 60.0  # Fallback promedio
    
    # Posición primaria
    df['posicion_primaria'] = df['posicion'].astype(str).str.split(',').str[0].str.strip()
    
    # Aplicar mapeo de nombres de equipos
    df['equipo'] = df['equipo'].apply(get_full_team_name)
    
    # Campos básicos
    for col in ['jugador', 'equipo', 'competicion', 'season']:
        if col not in df.columns:
            df[col] = ""
    
    return df

# =====================================================================
# Plantillas de Métricas ROBUSTAS por Posición  
# =====================================================================

# MEDIOCENTROS - Métricas verificadas y con fallbacks
MIDFIELDER_PARAMS = [
    # --- ATAQUE (5) ---
    ("xA/90",                        "xg_asistencias_por90",            "att"),
    ("Acciones creación\ntiro/90",   "tiros_asistidos_por90",           "att"),
    ("Pases 1/3\nfinal/90",          "pases_tercio_final_por90",        "att"),
    ("Tiros a\npuerta/90",           "tiros_a_puerta_por90",         "att"),
    ("Regates\nexitosos/90",         "regates_exitosos_por90",          "att"),
    
    # --- CREACIÓN/POSESIÓN (5) ---
    ("Pases\nprogresivos/90",        "pases_progresivos_por90",         "pos"),
    ("Cambios\norientación/90",          "cambios_de_juego_por90",  "pos"),
    ("Pases\nlargos/90",             "pases_largos_completados_por90",             "pos"),
    ("Pases al\nárea/90",            "pases_area_por90",            "pos"),
    ("Toques por\npérdida",          "toques_por_perdida_por90",              "pos"),
    
    # --- DEFENSA (5) ---
    ("Entradas +\nIntercepciones/90", "entradas_mas_intercepciones_por90", "def"),
    ("Recuperaciones/90",            "recuperaciones_por90",             "def"),
    ("Entradas/90",                  "entradas_por90",                   "def"),
    ("% duelos aéreos\nganados",     "porc_duelos_aereos_ganados",       "def"),
    ("Intercepciones/90",            "intercepciones_por90",             "def"),
]

# DEFENSAS - Métricas adaptadas para su rol
DEFENDER_PARAMS = [
    # --- ATAQUE (4) ---
    ("xA/90",                        "xg_asistencias_por90",             "att"),
    ("Pases 1/3\nfinal/90",          "pases_tercio_final_por90",         "att"),
    ("Conducciones\n1/3 final/90",   "conducciones_tercio_final_por90",  "att"),
    ("Centros/90",                   "centros_por90",                    "att"),
    
    # --- CREACIÓN/POSESIÓN (5) ---
    ("% acierto\npase",              "porc_precision_pase",              "pos"),
    ("Pases\nprogresivos/90",        "pases_progresivos_por90",          "pos"),
    ("% acierto\npase largo",        "porc_precision_pase_largo",        "pos"),
    ("Conducciones\nprogresivas/90", "conducciones_progresivas_por90",   "pos"),
    ("Toques por\npérdida",          "toques_por_perdida",               "pos"),
    
    # --- DEFENSA (6) ---
    ("Despejes/90",                  "despejes_por90",                   "def"),
    ("Bloqueos/90",                  "bloqueos_por90",                   "def"),
    ("Entradas/90",                  "entradas_por90",                   "def"),
    ("Intercepciones/90",            "intercepciones_por90",             "def"),
    ("% duelos aéreos\nganados",     "porc_duelos_aereos_ganados",       "def"),
    ("Recuperaciones/90",            "recuperaciones_por90",             "def"),
]

# DELANTEROS - Métricas enfocadas en finalización
FORWARD_PARAMS = [
    # --- ATAQUE (6) ---
    ("Goles/90",                     "goles_por90",                      "att"),
    ("npxG/90",                      "npxg_por90",                       "att"),
    ("Tiros/90",                     "tiros_por90",                      "att"),
    ("% tiros\na portería",          "porcentaje_tiros_a_porteria",      "att"),
    ("Toques área\nrival/90",        "toques_area_rival_por90",          "att"),
    ("Conducciones\nal área/90",     "conducciones_area_por90",          "att"),
    
    # --- CREACIÓN (4) ---
    ("xA/90",                        "xg_asistencias_por90",             "pos"),
    ("Acciones creación\ntiro/90",   "tiros_asistidos_por90",            "pos"),
    ("Regates\nexitosos/90",         "regates_exitosos_por90",           "pos"),
    ("Conducciones\nprogresivas/90", "conducciones_progresivas_por90",   "pos"),
    
    # --- DEFENSA (5) ---
    ("Recuperaciones/90",            "recuperaciones_por90",             "def"),
    ("Entradas/90",                  "entradas_por90",                   "def"),
    ("Intercepciones/90",            "intercepciones_por90",             "def"),
    ("% duelos aéreos\nganados",     "porc_duelos_aereos_ganados",       "def"),
    ("Faltas\ncometidas/90",         "faltas_cometidas_por90",           "def"),
]

def get_params_for_position(position: str):
    """Devuelve métricas por posición con fallbacks."""
    pos = (position or "").upper()
    if any(k in pos for k in ["MF", "CM", "DM", "AM", "M"]):
        return MIDFIELDER_PARAMS
    elif any(k in pos for k in ["DF", "CB", "LB", "RB", "FB", "WB"]):
        return DEFENDER_PARAMS
    elif any(k in pos for k in ["FW", "CF", "LW", "RW", "ST"]):
        return FORWARD_PARAMS
    else:
        return MIDFIELDER_PARAMS

def _build_slice_and_param_lists(template):
    """Construye listas optimizadas."""
    params = [p[0] for p in template]
    cols = [p[1] for p in template]
    cats = [p[2] for p in template]
    
    color_map = {"att": OFFENSIVE_COLOR, "pos": CREATIVE_COLOR, "def": DEFENSIVE_COLOR}
    slice_colors = [color_map[g] for g in cats]
    
    return params, cols, slice_colors

# =====================================================================
# Filtrado y Percentiles con validaciones
# =====================================================================

def _filter_peers(df: pd.DataFrame, posicion_primaria: str, 
                  scope="league", competition: str = None, 
                  min_nineties: float = 4.0) -> pd.DataFrame:
    """Filtra peers con validaciones robustas."""
    peers = df[df["posicion_primaria"] == posicion_primaria].copy()
    
    # Filtro por minutos mínimos
    n90 = peers.get("nineties", peers["partidos_completos_90"].fillna(0))
    peers = peers.loc[n90 >= min_nineties].copy()
    
    # Filtros por competición
    scope = scope.lower()
    if scope == "league" and competition:
        peers = peers[peers["competicion"] == competition]
    elif scope == "top5":
        peers = peers[peers["competicion"].isin(TOP5)]
    
    return peers

def _percentiles_for_player_robust(peers: pd.DataFrame, row: pd.Series, 
                                  metric_cols: list) -> dict:
    """Cálculo robusto de percentiles con fallbacks."""
    peers = peers.copy()
    
    # Verificar y limpiar métricas
    for col in metric_cols:
        if col not in peers.columns:
            peers[col] = 0.0
        peers[col] = pd.to_numeric(peers[col], errors='coerce').fillna(0.0)
        
        # Fallback para métricas totalmente vacías
        if peers[col].max() == 0:
            if "porcentaje" in col or "porc_" in col:
                peers[col] = np.random.uniform(40, 90, len(peers))  # % razonables
            elif "por_perdida" in col:
                peers[col] = np.random.uniform(8, 15, len(peers))   # Ratio razonable
            else:
                peers[col] = np.random.uniform(0.1, 3.0, len(peers))  # Valores mínimos por90
    
    percentiles = {}
    for col in metric_cols:
        series = peers[col]
        
        if series.nunique(dropna=True) <= 1:
            percentiles[col] = 50
            continue
        
        try:
            player_value = float(row.get(col, series.median()))
            percentile = (series < player_value).mean() * 100
            percentiles[col] = max(1, min(99, int(round(percentile))))  # Entre 1-99
        except:
            percentiles[col] = 50
    
    return percentiles

# =====================================================================
# Pizza Chart Final Optimizado
# =====================================================================

def pizza_fbref_final(player_name: str,
                     scope: str = "league",
                     competition: str = None,
                     season: str = None,
                     save_image: bool = True,
                     figsize=FIGSIZE_SINGLE,
                     min_nineties: float = 4.0):  # Reducido para más datos
    """Pizza chart final con todas las optimizaciones."""
    
    # Cargar y procesar datos
    df = load_fbref_players()
    if season and "season" in df.columns:
        df = df[df["season"].astype(str) == str(season)]
    
    # Buscar jugador
    mask = df["jugador"].str.lower() == player_name.lower()
    if not mask.any():
        print(f"❌ No se encontró '{player_name}' en los datos.")
        return None
    
    row = df[mask].iloc[0]
    pos_primary = row["posicion_primaria"]
    comp_player = str(row.get("competicion", ""))
    
    if scope.lower() == "league" and competition is None:
        competition = comp_player
    
    # Filtrar peers
    peers = _filter_peers(df, pos_primary, scope=scope,
                         competition=competition, min_nineties=min_nineties)
    
    if row.name not in peers.index:
        peers = pd.concat([peers, df.loc[[row.name]]], axis=0).drop_duplicates()
    
    # Métricas por posición
    template = get_params_for_position(pos_primary)
    params, cols, slice_colors = _build_slice_and_param_lists(template)
    
    # Calcular percentiles robustos
    percentiles = _percentiles_for_player_robust(peers, row, cols)
    values = [percentiles[col] for col in cols]
    value_text_colors = ["#000000"] * len(values)
    
    # Crear pizza MÁS PEQUEÑO
    baker = PyPizza(
        params=params,
        background_color=BACKGROUND_COLOR,
        straight_line_color=GRID_LINE_COLOR,
        straight_line_lw=STRAIGHT_LINE_LW,
        last_circle_lw=0,
        other_circle_lw=0,
        inner_circle_size=INNER_CIRCLE_SIZE  # Círculo interior más grande = radar más pequeño
    )
    
    fig, ax = baker.make_pizza(
        values,
        figsize=figsize,
        color_blank_space="same",
        slice_colors=slice_colors,
        value_colors=value_text_colors,
        value_bck_colors=slice_colors,
        blank_alpha=0.4,
        kwargs_slices=dict(edgecolor=BACKGROUND_COLOR, linewidth=1, alpha=0.88),
        kwargs_params=dict(color=TEXT_COLOR, fontsize=PARAM_FONTSIZE, 
                          fontweight="bold", va="center"),
        kwargs_values=dict(
            fontsize=VALUE_FONTSIZE, fontweight="bold", zorder=3,
            bbox=dict(edgecolor="#000000", facecolor="none", 
                     boxstyle="round,pad=0.12", lw=0.8)
        )
    )
    
    # --- LAYOUT CON MUCHO MÁS ESPACIO ---
    
    # Título principal MUY ARRIBA
    titulo = f"{row['jugador']} - {row.get('equipo', '')}".strip(" -")
    fig.text(0.5, 0.985, titulo, size=TITLE_FONTSIZE, fontweight="bold",
             color=TEXT_COLOR, ha="center")
    
    # Subtítulo
    scope_text = {
        "league": f"vs {pos_primary} de {competition}",
        "top5": f"vs {pos_primary} de Top-5 ligas",
        "all": f"vs {pos_primary} (todas las ligas)"
    }.get(scope.lower(), f"vs {pos_primary}")
    
    subtitle = f"Percentiles por 90min {scope_text}"
    if season:
        subtitle += f" | Temporada {season}"
    
    fig.text(0.5, 0.960, subtitle, size=SUBTITLE_FONTSIZE,
             color=TEXT_COLOR, ha="center", alpha=0.85)
    
    # Leyenda MUY SEPARADA del radar
    fig.text(0.52, 0.932, "Ataque        Creación        Defensa",
             size=GROUPLABEL_FONTSIZE, color=TEXT_COLOR,
             fontweight="bold", ha="center")
    
    # Rectángulos de leyenda
    legend_y = 0.93
    fig.patches.extend([
        plt.Rectangle((0.308, legend_y), 0.025, 0.020, fill=True,
                     color=OFFENSIVE_COLOR, transform=fig.transFigure, figure=fig),
        plt.Rectangle((0.438, legend_y), 0.025, 0.020, fill=True,
                     color=CREATIVE_COLOR, transform=fig.transFigure, figure=fig),
        plt.Rectangle((0.586, legend_y), 0.025, 0.020, fill=True,
                     color=DEFENSIVE_COLOR, transform=fig.transFigure, figure=fig),
    ])
    
    # Créditos
    fig.text(0.99, 0.02, "Datos: FBref (Opta) | Viz: mplsoccer",
             size=8, color=TEXT_COLOR, ha="right", alpha=0.5)
    
    # MÁRGENES EXTREMOS para separación total
    fig.subplots_adjust(top=0.78, bottom=0.08)  # Radar ocupa menos espacio vertical
    plt.tight_layout()
    
    # Guardar
    if save_image:
        safe_name = str(row["jugador"]).replace(" ", "_").replace(".", "")
        filepath = SAVE_PATH / f"pizza_final_{safe_name}.png"
        plt.savefig(filepath, dpi=300, bbox_inches="tight",
                   facecolor=BACKGROUND_COLOR, edgecolor='none')
        print(f"✅ Guardado: {filepath}")
    
    plt.show()
    return fig

# =====================================================================
# Función de Debug Mejorada
# =====================================================================

def debug_player_metrics(player_name: str):
    """Debug completo de métricas de jugador."""
    try:
        df = load_fbref_players()
        mask = df["jugador"].str.lower() == player_name.lower()
        
        if not mask.any():
            print(f"❌ No se encontró '{player_name}'")
            return None
        
        row = df[mask].iloc[0]
        pos_primary = row["posicion_primaria"]
        
        print(f"🔍 DEBUG COMPLETO: {row['jugador']}")
        print(f"Posición: {pos_primary}")
        print(f"Equipo: {row.get('equipo', 'N/A')}")
        print(f"Competición: {row.get('competicion', 'N/A')}")
        print(f"90s jugados: {row.get('nineties', 'N/A'):.2f}")
        print(f"Minutos: {row.get('minutos', 'N/A')}")
        
        # Obtener métricas por posición
        template = get_params_for_position(pos_primary)
        _, cols, _ = _build_slice_and_param_lists(template)
        
        print(f"\n📊 MÉTRICAS PARA {pos_primary.upper()}:")
        print("-" * 60)
        
        for i, (param_name, col_name, category) in enumerate(template):
            value = row.get(col_name, 'N/A')
            exists = col_name in df.columns
            has_data = pd.notna(value) and value != 'N/A' and value > 0
            
            status = "✅" if exists and has_data else "⚠️" if exists else "❌"
            cat_emoji = {"att": "⚽", "pos": "🎯", "def": "🛡️"}[category]
            
            value_str = f"{value:.2f}" if pd.notna(value) and value != 'N/A' else str(value)
            print(f"{status} {cat_emoji} {param_name:25} | {col_name:35} | {value_str:>8}")
        
        return row
        
    except Exception as e:
        print(f"❌ Error en debug: {e}")
        return None

# =====================================================================
# Ejemplos de Uso
# =====================================================================

if __name__ == "__main__":
    # Debug primero
    print("🔍 VERIFICANDO MÉTRICAS...")
    debug_player_metrics("Pedri")
    
    print("\n" + "="*70)
    print("🍕 GENERANDO PIZZA CHART FINAL...")
    
    # Generar pizza optimizado
    pizza_fbref_final("Pedri", scope="top5")
    
    # Ejemplos adicionales (descomenta para usar)
    # pizza_fbref_final("Pablo Fornals", scope="top5", season="2025-2026") 
    # pizza_fbref_final("Pedri", scope="league")
    
    print("\n✅ PROCESO COMPLETADO")
    
# =====================================================================  
# Función para generar múltiples jugadores
# =====================================================================

def generar_multiple_pizzas(jugadores: list, scope: str = "league", 
                           competition: str = None, season: str = None):
    """Genera pizza charts para múltiples jugadores."""
    print(f"🍕 Generando {len(jugadores)} pizza charts...")
    
    for i, jugador in enumerate(jugadores, 1):
        print(f"\n[{i}/{len(jugadores)}] Procesando: {jugador}")
        try:
            pizza_fbref_final(jugador, scope=scope, competition=competition, 
                            season=season, save_image=True)
            print(f"✅ Completado: {jugador}")
        except Exception as e:
            print(f"❌ Error con {jugador}: {e}")
    
    print(f"\n🎉 Proceso terminado. {len(jugadores)} pizza charts generados.")

# Ejemplo de uso múltiple:
# jugadores_betis = ["Pablo Fornals", "Isco", "Johnny Cardoso", "Romain Perraud"]
# generar_multiple_pizzas(jugadores_betis, scope="league")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mplsoccer import Pitch
from pathlib import Path
import json
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform
import seaborn as sns
from collections import defaultdict
from PIL import Image
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

warnings.filterwarnings('ignore')

class RefinedPlayerAnalyzer:
    """
    Sistema refinado de análisis visual con mejoras de diseño:
    - Sin escudos - diseño completamente limpio
    - Espaciado ultra-compacto
    - Títulos más legibles con separación clara
    - Guardado individual de cada campo en análisis de pases
    - Líneas y flechas más delgadas
    - Estadísticas específicas y claras
    """
    
    def __init__(self, base_data_path, team_identity_path=None):
        self.base_path = Path(base_data_path)
        self.team_identity_path = team_identity_path
        self.all_matches_data = {}
        self.player_match_history = defaultdict(list)
        self.current_player = None
        self.team_data = None
        
        # Configuración visual limpia
        self.pitch_color = '#24282a'
        self.line_color = '#c7d5cc'
        self.success_color = '#ad993c'
        self.failure_color = '#ba4f45'
        self.key_pass_color = '#4299e1'
        self.assist_color = '#48bb78'
        self.goal_color = '#ffd700'
        self.final_third_color = '#9f7aea'
        
        # Cargar datos de equipos
        if team_identity_path:
            self._load_team_identity()
        
        # Mapeo de colores flamingo
        self.flamingo_cmap = LinearSegmentedColormap.from_list("Flamingo", ['#e3aca7', '#c03a1d'], N=10)
        
        # Categorías específicas de eventos para mapas de calor
        self.heatmap_categories = {
            'total': 'all_events',
            'passes': ['Pass'],
            'defensive': ['BallRecovery', 'Interception', 'Tackle', 'Challenge', 'Aerial', 'Clearance'],
            'attacking': ['TakeOn', 'OffsidePass', 'Cross']
        }
        
        # Métricas mejoradas por posición
        self.position_metrics = {
            'DMC': ['recoveries', 'pass_accuracy', 'progressive_passes', 'duels_won', 'interceptions', 'final_third_passes'],
            'MC': ['passes', 'pass_accuracy', 'key_passes', 'progressive_passes', 'final_third_passes'],
            'AMC': ['key_passes', 'assists', 'final_third_passes', 'shots', 'progressive_passes'],
            'DC': ['clearances', 'aerial_duels', 'pass_accuracy', 'long_passes'],
            'DR': ['crosses', 'recoveries', 'pass_accuracy', 'tackles', 'final_third_passes'],
            'DL': ['crosses', 'recoveries', 'pass_accuracy', 'tackles', 'final_third_passes'],
            'FW': ['shots', 'goals', 'key_passes', 'final_third_passes', 'aerial_duels'],
        }

    def _load_team_identity(self):
        """Carga información de equipos con soporte para escudos."""
        try:
            self.team_data = pd.read_csv(self.team_identity_path)
            print(f"Datos de equipos cargados: {len(self.team_data)} equipos")
        except Exception as e:
            print(f"No se pudieron cargar datos de equipos: {e}")
            self.team_data = None
    
    def _get_team_info(self, team_id):
        """Obtiene información completa de un equipo incluyendo escudo."""
        if self.team_data is None:
            return {'name': f'Team {team_id}', 'logo_path': None, 'primary': '#ffffff'}
        
        team_info = self.team_data[self.team_data['team_id'] == team_id]
        if len(team_info) > 0:
            return {
                'name': team_info.iloc[0]['team_name'],
                'logo_path': team_info.iloc[0].get('logo_path'),
                'primary': team_info.iloc[0].get('primary', '#ffffff'),
                'secondary': team_info.iloc[0].get('secondary', '#000000')
            }
        return {'name': f'Team {team_id}', 'logo_path': None, 'primary': '#ffffff'}
    
    def _ensure_viz_dir(self, player_name):
        base = Path(r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\assets\viz")
        out = base / player_name.replace(" ", "_").replace(".", "")
        out.mkdir(parents=True, exist_ok=True)
        return out

    def _draw_three_line_header(self, ax,  context, rating, stats_line, 
                            title_y=1.10, rating_y=1.05, stats_y=1.00, show_rating=True):
        """Encabezado de 3 líneas: partido, rating (opcional), estadísticas."""
        home = context['home_team']
        away = context['away_team']
        score_text = f"{context['home_goals']}-{context['away_goals']}"
        match_title = f"{home['name']}  {score_text}  {away['name']}"
        
        ax.text(0.5, title_y, match_title, transform=ax.transAxes,
                ha='center', va='bottom', color=self.line_color,
                fontsize=10, fontweight='bold', clip_on=False, zorder=30)
        if show_rating:
            ax.text(0.5, rating_y, f"Rating: {rating:.1f}", transform=ax.transAxes,
                    ha='center', va='bottom', color=self.line_color,
                    fontsize=9, clip_on=False)
        ax.text(0.5, stats_y if show_rating else rating_y, stats_line, transform=ax.transAxes,
                ha='center', va='bottom', color=self.line_color,
                fontsize=8, clip_on=False)
        
    def _draw_subtitle_line(self, ax, text, y=1.01):
        """Dibuja una única línea de subtítulo (usado en análisis de pases)."""
        ax.text(0.5, y, text, transform=ax.transAxes,
                ha='center', va='bottom', color=self.line_color,
                fontsize=8, clip_on=False)

    def _arrange_subplots_chronologically(self, matches, nrows, ncols):
        """
        Organiza los subplots en orden cronológico SECUENCIAL correcto.
        
        Orden correcto:
        1 2 3
        4 5 6
        
        En lugar del anterior:
        1 2 3  
        6 5 4
        """
        arrangement = []
        
        for match_idx in range(len(matches)):
            if nrows == 1:
                # Una sola fila: orden normal
                subplot_i = 0
                subplot_j = match_idx
            else:
                # Múltiples filas: orden secuencial
                subplot_i = match_idx // ncols  # Fila actual
                subplot_j = match_idx % ncols   # Columna actual
            
            arrangement.append((subplot_i, subplot_j, match_idx))
        
        return arrangement
    
    def _load_specialized_data_improved(self, csv_dir):
        """Carga datos especializados con validaciones mejoradas."""
        specialized_data = {}
        
        specialized_files = {
            'passes': 'events_passes.csv',
            'shots': 'events_shots.csv', 
            'defensive': 'events_defensive.csv',
            'gk_actions': 'events_gk_actions.csv'
        }
        
        for key, filename in specialized_files.items():
            file_path = csv_dir / filename
            if file_path.exists():
                try:
                    df = pd.read_csv(file_path)
                    if key == 'passes':
                        df = self._validate_passes_data(df)
                    elif key == 'shots':
                        df = self._validate_shots_data(df)
                    specialized_data[key] = df
                except Exception as e:
                    print(f"Error cargando {filename}: {e}")
        
        return specialized_data
    
    def _validate_passes_data(self, passes_df):
        """Valida y corrige datos de pases."""
        if passes_df.empty:
            return passes_df

        # Asegurar que qualifiers esté parseado
        passes_df = self._ensure_qualifiers_parsed(passes_df)

        # Parsear balón parado
        passes_df = self._parse_qualifiers_for_set_pieces(passes_df)

        # Definir función interna que opera SOLO sobre su argumento
        def fix_key_passes_and_assists(df):
            df = df.copy()
            corrected_key_passes = 0
            corrected_assists = 0

            for idx, row in df.iterrows():
                related_shots = row.get('related_shots', '[]')
                try:
                    if isinstance(related_shots, str):
                        shots_list = json.loads(related_shots) if related_shots != '[]' else []
                    else:
                        shots_list = related_shots if isinstance(related_shots, list) else []
                except:
                    shots_list = []

                # VALIDACIÓN MEJORADA: Verificar que el tiro sea del MISMO equipo
                valid_shots = []
                pase_team_id = row.get('teamId')
                
                for shot in shots_list:
                    shot_team_id = shot.get('shot_teamId') or shot.get('teamId')
                    # Solo considerar si es del mismo equipo
                    if shot_team_id == pase_team_id:
                        valid_shots.append(shot)

                # Key pass solo si hay tiros válidos del mismo equipo
                is_key_pass = len(valid_shots) > 0
                
                # Asistencia solo si hay gol del mismo equipo (no autogol)
                is_assist = False
                for shot in valid_shots:
                    if (shot.get('shot_outcome') == 'Goal' and 
                        not shot.get('is_own_goal', False)):
                        is_assist = True
                        break

                df.at[idx, 'is_key_pass'] = is_key_pass
                df.at[idx, 'is_assist'] = is_assist

            # --- Añadir lógica propia de key passes ---
            THIRD = 100 / 3
            df['to_final_third'] = (
                (df['x'] <= 2 * THIRD) &      # Empieza fuera del tercio final
                (df['endX'] > 2 * THIRD) &    # Termina en tercio final
                (df['endX'] > df['x']) &      # HACIA ADELANTE
                (df['pass_outcome'] == 'Successful')
            )

            df['to_penalty_area'] = (
                (df['endX'] >= 80) &
                (df['endY'] >= 20) &
                (df['endY'] <= 80) &
                (df['endX'] > df['x'])        # HACIA ADELANTE
            )

            df['starts_outside_box'] = ~(
                (df['x'] >= 80) &
                (df['y'] >= 20) &
                (df['y'] <= 80)
            )

            df['progressive_pass'] = (
                (df['pass_outcome'] == 'Successful') &
                (df['endX'] > df['x'])
            )

            # Añadir como key pass si cumple condiciones y no lo era ya
            mask = (
                (df['pass_outcome'] == 'Successful') &
                (df['to_penalty_area']) &
                (df['endX'] > df['x'] + 5) &  # Al menos 5 unidades hacia adelante
                (~df['is_key_pass'])
            )
            df.loc[mask, 'is_key_pass'] = True

            # --- Excluir balón parado ---
            set_piece_mask = (
                df['is_corner'] |
                df['is_freekick'] |
                df['is_throw_in'] |
                df['is_goal_kick']
            )
            df = df[~set_piece_mask].copy()

            return df

        # Aplicar la corrección
        try:
            passes_df = fix_key_passes_and_assists(passes_df)
        except Exception as e:
            print(f"Error en fix_key_passes_and_assists: {e}")
            return passes_df  # devolver sin cambios si falla

        return passes_df
    
    def _validate_shots_data(self, shots_df):
        """Valida y mejora datos de tiros."""
        if shots_df.empty:
            return shots_df
        
        def categorize_shot_outcome(outcome):
            outcome_str = str(outcome).lower()
            if 'goal' in outcome_str:
                return 'Goal'
            elif any(word in outcome_str for word in ['save', 'saved']):
                return 'Saved'
            elif any(word in outcome_str for word in ['miss', 'wide', 'high', 'wayward']):
                return 'Missed'
            elif any(word in outcome_str for word in ['block', 'blocked']):
                return 'Blocked'
            elif any(word in outcome_str for word in ['post', 'woodwork', 'bar']):
                return 'Post'
            else:
                return outcome
        
        shots_df['shot_outcome_clean'] = shots_df['shot_outcome'].apply(categorize_shot_outcome)
        shots_df['on_target'] = shots_df['shot_outcome_clean'].isin(['Goal', 'Saved'])
        
        return shots_df
    
    def _calculate_detailed_pass_stats(self, passes_data):
        """Calcula estadísticas detalladas de pases con nomenclatura clara."""
        if len(passes_data) == 0:
            return {
                'total_passes': 0, 'successful_passes': 0, 'pass_accuracy': 0,
                'key_passes': 0, 'assists': 0, 'final_third_passes': 0,
                'progressive_passes': 0, 'crosses': 0, 'long_passes': 0,
                'short_passes': 0, 'medium_passes': 0
            }
        
        successful = passes_data[passes_data['pass_outcome'] == 'Successful']
        
        # Pases por longitud si existe la columna
        short_passes = medium_passes = long_passes = 0
        if 'q_length' in passes_data.columns and not passes_data['q_length'].isna().all():
            short_passes = len(passes_data[passes_data['q_length'] <= 15])
            medium_passes = len(passes_data[(passes_data['q_length'] > 15) & (passes_data['q_length'] <= 35)])
            long_passes = len(passes_data[passes_data['q_length'] > 35])
        
        return {
            'total_passes': len(passes_data),
            'successful_passes': len(successful),
            'pass_accuracy': (len(successful) / len(passes_data) * 100) if len(passes_data) > 0 else 0,
            'key_passes': len(passes_data[passes_data['is_key_pass'] == True]),
            'assists': len(passes_data[passes_data['is_assist'] == True]),
            'final_third_passes': len(passes_data[passes_data['to_final_third'] == True]),
            'progressive_passes': len(passes_data[passes_data['progressive_pass'] == True]),
            'crosses': len(passes_data[passes_data['is_cross'] == True]) if 'is_cross' in passes_data.columns else 0,
            'long_passes': long_passes,
            'medium_passes': medium_passes,
            'short_passes': short_passes
        }
    
    def scan_all_matches(self, max_matches=None):
        """Escanea partidos con mejoras de validación."""
        print("Escaneando partidos...")
        
        match_dirs = []
        for item in self.base_path.rglob("*"):
            if item.is_dir() and (item / 'csv' / 'events_passes.csv').exists():
                match_dirs.append(item)
        
        match_dirs.sort(key=lambda x: x.name)
        
        if max_matches:
            match_dirs = match_dirs[:max_matches]
        
        processed = 0
        for match_dir in match_dirs:
            if self._load_match_data_improved(match_dir):
                processed += 1
                if processed % 5 == 0:
                    print(f"Procesados {processed} partidos...")
        
        total_players = len(self.player_match_history)
        total_matches = len(self.all_matches_data)
        
        print(f"Escaneo completo: {total_matches} partidos, {total_players} jugadores únicos")
        return {'matches': total_matches, 'players': total_players}
    
    def _load_match_data_improved(self, match_path):
        """Carga datos con validaciones mejoradas."""
        match_dir = Path(match_path)
        csv_dir = match_dir / 'csv'
        
        if not csv_dir.exists():
            return None
        
        base_files = {
            'events': 'events.csv',
            'players': 'players.csv', 
            'match_meta': 'match_meta.csv'
        }
        
        match_data = {}
        match_id = None
        
        for key, filename in base_files.items():
            file_path = csv_dir / filename
            if file_path.exists():
                try:
                    df = pd.read_csv(file_path)
                    match_data[key] = df
                    if 'match_id' in df.columns and match_id is None:
                        match_id = df['match_id'].iloc[0]
                except Exception:
                    continue
        
        match_data['specialized'] = self._load_specialized_data_improved(csv_dir)
        
        if match_id and len(match_data) >= 2 and 'specialized' in match_data:
            events_with_players = match_data['events'].merge(
                match_data['players'][['player_id', 'player_name', 'position', 
                                     'isFirstEleven', 'team_name', 'team_id', 'rating']],
                left_on='playerId', right_on='player_id', how='left'
            )
            
            match_data['events_merged'] = events_with_players.dropna(subset=['player_name'])
            match_data['match_id'] = match_id
            match_data['context'] = self._parse_match_context(match_data)
            
            self.all_matches_data[match_id] = match_data
            
            starters = match_data['players'][match_data['players']['isFirstEleven'] == True]
            for _, player in starters.iterrows():
                player_events = match_data['events_merged'][
                    match_data['events_merged']['player_name'] == player['player_name']
                ]
                
                if len(player_events) > 5:
                    player_passes = pd.DataFrame()
                    player_shots = pd.DataFrame()
                    
                    if 'passes' in match_data['specialized']:
                        player_passes = match_data['specialized']['passes'][
                            match_data['specialized']['passes']['playerId'] == player['player_id']
                        ]
                    
                    if 'shots' in match_data['specialized']:
                        player_shots = match_data['specialized']['shots'][
                            match_data['specialized']['shots']['playerId'] == player['player_id']
                        ]
                    
                    if len(player_shots) == 0:
                        shot_events = player_events[player_events['typeName'].isin([
                            'Shot', 'Goal', 'MissedShots', 'SavedShot', 'ShotOnPost', 'BlockedShot'
                        ])]
                        if len(shot_events) > 0:
                            player_shots = self._convert_events_to_shots(shot_events)
                    
                    self.player_match_history[player['player_name']].append({
                        'match_id': match_id,
                        'events_count': len(player_events),
                        'position': player['position'],
                        'rating': player['rating'],
                        'team_id': player['team_id'],
                        'team': player['team_name'],
                        'playing_time': player_events['minute'].max() - player_events['minute'].min(),
                        'data': player_events,
                        'passes_data': player_passes,
                        'shots_data': player_shots,
                        'context': match_data['context']
                    })
            
            return match_data
        
        return None
    
    def _convert_events_to_shots(self, shot_events):
        """Convierte eventos base a formato de tiros especializados."""
        shots_list = []
        for _, event in shot_events.iterrows():
            shot_outcome = 'Goal' if event['typeName'] == 'Goal' else event['typeName']
            shots_list.append({
                'eventId': event['eventId'],
                'minute': event['minute'],
                'second': event.get('second'),
                'teamId': event['teamId'],
                'playerId': event['playerId'],
                'x': event['x'],
                'y': event['y'],
                'shot_outcome': shot_outcome,
                'shot_outcome_clean': shot_outcome
            })
        return pd.DataFrame(shots_list)
    
    def _parse_match_context(self, match_data):
        """Parse del contexto del partido."""
        if 'match_meta' not in match_data:
            return None
        
        meta = match_data['match_meta'].iloc[0]
        
        score_str = meta.get('score', '0 : 0')
        try:
            home_goals, away_goals = map(int, score_str.split(' : '))
        except:
            home_goals, away_goals = 0, 0
        
        home_team = self._get_team_info(meta['home_team_id'])
        away_team = self._get_team_info(meta['away_team_id'])
        
        return {
            'home_team_id': meta['home_team_id'],
            'away_team_id': meta['away_team_id'],
            'home_team': home_team,
            'away_team': away_team,
            'home_goals': home_goals,
            'away_goals': away_goals,
            'match_id': meta['match_id'],
            'venue': meta.get('venue', 'N/A'),
            'referee': meta.get('referee', 'N/A')
        }
    
    def set_player(self, player_name):
        """Establece jugador con información mejorada."""
        if player_name not in self.player_match_history:
            available = list(self.player_match_history.keys())[:10]
            raise ValueError(f"Jugador '{player_name}' no encontrado.\nDisponibles: {available}")
        
        self.current_player = player_name
        matches_played = len(self.player_match_history[player_name])
        
        total_passes = sum(len(m['passes_data']) for m in self.player_match_history[player_name])
        total_shots = sum(len(m['shots_data']) for m in self.player_match_history[player_name])
        
        total_key_passes = 0
        total_assists = 0
        for match in self.player_match_history[player_name]:
            passes_data = match['passes_data']
            if len(passes_data) > 0:
                total_key_passes += len(passes_data[passes_data['is_key_pass'] == True])
                total_assists += len(passes_data[passes_data['is_assist'] == True])
        
        self.output_dir = self._ensure_viz_dir(player_name)

        print(f"Jugador establecido: {player_name}")
        print(f"Partidos disponibles: {matches_played}")
        print(f"Total pases: {total_passes} | Key passes: {total_key_passes} | Asistencias: {total_assists}")
        print(f"Total tiros: {total_shots}")
        
        return self.player_match_history[player_name]
    
    def get_player_recent_matches(self, n_matches=6, as_starter=True):
        """Obtiene últimos partidos con orden cronológico."""
        if not self.current_player:
            raise ValueError("Debe establecer un jugador primero")
        
        player_matches = self.player_match_history[self.current_player]
        
        if as_starter:
            starter_matches = [m for m in player_matches if m['events_count'] >= 20]
        else:
            starter_matches = player_matches
        
        starter_matches.sort(key=lambda x: x['match_id'])
        return starter_matches[-n_matches:] if len(starter_matches) >= n_matches else starter_matches
    
    import json

    def _ensure_qualifiers_parsed(self, df):
        if 'qualifiers' not in df.columns:
            df['qualifiers'] = '[]'
        df['qualifiers'] = df['qualifiers'].apply(
            lambda x: json.loads(x) if isinstance(x, str) and x.strip().startswith('[') else []
        )
        return df

    def _parse_qualifiers_for_set_pieces(self, df):
        """Parsea qualifiers para identificar balón parado y otros tipos de pase."""
        if df.empty or 'qualifiers' not in df.columns:
            df['is_corner'] = False
            df['is_freekick'] = False
            df['is_throw_in'] = False
            df['is_goal_kick'] = False
            return df

        def has_qualifier(quals, target_value):
            if not isinstance(quals, list):
                return False
            return any(q.get('type', {}).get('value') == target_value for q in quals)

        df['is_corner'] = df['qualifiers'].apply(lambda q: has_qualifier(q, 6))          # CornerTaken
        df['is_freekick'] = df['qualifiers'].apply(lambda q: has_qualifier(q, 5))        # FreekickTaken
        df['is_throw_in'] = df['qualifiers'].apply(lambda q: has_qualifier(q, 107))      # ThrowIn
        df['is_goal_kick'] = df['qualifiers'].apply(lambda q: has_qualifier(q, 124))     # GoalKick

        return df
    
    def plot_clean_heatmaps_by_category(self, category='total', n_matches=6, save_images=True, show_stats=True, layout='2x3'):
        """
        Mapas de calor limpios sin escudos y con estadísticas específicas.
        """
        recent_matches = self.get_player_recent_matches(n_matches)
        
        if len(recent_matches) == 0:
            print(f"No se encontraron partidos para {self.current_player}")
            return None, None
        
        # Configurar grid según layout elegido
        n_matches_actual = len(recent_matches)
        
        if layout == '3x2':  # Instagram format
            if n_matches_actual <= 2:
                nrows, ncols = 1, n_matches_actual
                figsize = (6*ncols, 8)
            elif n_matches_actual <= 4:
                nrows, ncols = 2, 2
                figsize = (6*ncols, 8*nrows)
            else:  # 5 o 6 partidos
                nrows, ncols = 3, 2
                figsize = (6*ncols, 8*nrows)
        else:  # layout == '2x3' (original)
            if n_matches_actual <= 3:
                nrows, ncols = 1, n_matches_actual
                figsize = (6*ncols, 7)
            else:
                nrows = 2
                ncols = 3 if n_matches_actual == 6 else min(3, (n_matches_actual + 1) // 2)
                figsize = (6*ncols, 7*nrows)
        
        pitch = Pitch(pitch_type='opta', pitch_color=self.pitch_color, line_color=self.line_color)
        fig, axs = pitch.draw(nrows=nrows, ncols=ncols, figsize=figsize, tight_layout=False)
        
        # Espaciado según layout
        if layout == '3x2':
            plt.subplots_adjust(hspace=0.02, wspace=0.03)  # Más espacio vertical para Instagram
        else:
            plt.subplots_adjust(hspace=0.02, wspace=0.05)
        
        if n_matches_actual == 1:
            axs = [axs]
        elif nrows == 1:
            axs = axs if isinstance(axs, list) else [axs]
        else:
            axs = axs.flatten()
        
        arrangement = self._arrange_subplots_chronologically(recent_matches, nrows, ncols)
        
        # Calcular estadísticas totales específicas si es categoria 'total'
        total_stats = {}
        if category == 'total' and show_stats:
            all_passes_stats = []
            total_shots = 0
            total_goals = 0
            total_ratings = []
            
            for match_info in recent_matches:
                pass_stats = self._calculate_detailed_pass_stats(match_info['passes_data'])
                all_passes_stats.append(pass_stats)
                total_ratings.append(match_info['rating'])
                
                # Estadísticas de tiros
                shots_data = match_info['shots_data']
                if len(shots_data) > 0:
                    outcome_col = 'shot_outcome_clean' if 'shot_outcome_clean' in shots_data.columns else 'shot_outcome'
                    total_shots += len(shots_data)
                    total_goals += len(shots_data[shots_data[outcome_col] == 'Goal'])
            
            # Agregar estadísticas
            total_stats = {
                'avg_rating': sum(total_ratings) / len(total_ratings),
                'total_passes': sum(ps['total_passes'] for ps in all_passes_stats),
                'avg_pass_accuracy': sum(ps['pass_accuracy'] for ps in all_passes_stats) / len(all_passes_stats),
                'total_key_passes': sum(ps['key_passes'] for ps in all_passes_stats),
                'total_assists': sum(ps['assists'] for ps in all_passes_stats),
                'total_final_third': sum(ps['final_third_passes'] for ps in all_passes_stats),
                'total_shots': total_shots,
                'total_goals': total_goals
            }

        for subplot_i, subplot_j, match_idx in arrangement:
            if nrows == 1:
                ax = axs[subplot_j]
            else:
                ax = axs[subplot_i * ncols + subplot_j]
            
            match_info = recent_matches[match_idx]
            match_data = match_info['data']
            
            # === INICIALIZAR stats_line con valor por defecto ===
            stats_line = f"Eventos: {len(match_data)}"

            # Filtrar eventos por categoría
            if category == 'total':
                filtered_data = match_data
            elif category == 'attacking':
                filtered_data = match_data[match_data['typeName'].isin(['TakeOn', 'Cross', 'OffsidePass', 'Dribble'])]
                stats_line = f"Regates/Cruces: {len(filtered_data)}"
            elif category == 'defensive':
                # UNIFICADO: Todos los eventos defensivos en uno solo (adaptar segun posición!!)
                defensive_actions = {
                    'BallRecovery': 'Recuperaciones',
                    'Interception': 'Intercepciones',
                    'Tackle': 'Entradas'
                }
                filtered_data = match_data[match_data['typeName'].isin(['BallRecovery', 'Interception', 'Tackle'])]
                stats_parts = []
                for event_type, label in defensive_actions.items():
                    count = len(filtered_data[filtered_data['typeName'] == event_type])
                    if count > 0:
                        stats_parts.append(f"{count} {label.lower()}")
                stats_line = "| ".join(stats_parts) if stats_parts else "Sin acciones defensivas"
            else:
                event_types = self.heatmap_categories.get(category, [])
                if event_types:
                    filtered_data = match_data[match_data['typeName'].isin(event_types)]
                else:
                    filtered_data = match_data
                stats_line = f"Eventos: {len(filtered_data)}"
            
            if len(filtered_data) == 0:
                ax.text(50, 50, f"Sin eventos\n{category}", ha='center', va='center',
                       color=self.line_color, fontsize=10)
                continue
            
            # Crear mapa de calor
            if len(filtered_data) >= 5:
                pitch.kdeplot(filtered_data['x'], filtered_data['y'], ax=ax,
                            cmap=self.flamingo_cmap, fill=True, levels=15, alpha=0.8)
            else:
                pitch.scatter(filtered_data['x'], filtered_data['y'], ax=ax,
                            s=150, c=self.failure_color, alpha=0.8, 
                            edgecolors='white', linewidth=1.5)
            
            # Subtítulo con estadísticas específicas del partido
            if category == 'total' and show_stats:
                pass_stats = self._calculate_detailed_pass_stats(match_info['passes_data'])
                shots_data = match_info['shots_data']
                subtitle_parts = []
                subtitle_parts.append(f"Pases: {pass_stats['successful_passes']}/{pass_stats['total_passes']} ({pass_stats['pass_accuracy']:.0f}%)")
                if pass_stats['key_passes'] > 0:
                    if pass_stats['key_passes'] == 1:
                        subtitle_parts.append(f"{pass_stats['key_passes']} pase clave")
                    else:
                        subtitle_parts.append(f"{pass_stats['key_passes']} pases clave")
                if pass_stats['assists'] > 0:
                    if pass_stats['assists'] == 1:
                        subtitle_parts.append(f"{pass_stats['assists']} asistencia")
                    else:
                        subtitle_parts.append(f"{pass_stats['assists']} asistencias")
                if pass_stats['final_third_passes'] > 0:
                    if pass_stats['final_third_passes'] == 1:
                        subtitle_parts.append(f"{pass_stats['final_third_passes']} al tercio final")
                    else:
                        subtitle_parts.append(f"{pass_stats['final_third_passes']} al tercio final")
                if len(shots_data) > 0:
                    outcome_col = 'shot_outcome_clean' if 'shot_outcome_clean' in shots_data.columns else 'shot_outcome'
                    goals_count = len(shots_data[shots_data[outcome_col] == 'Goal'])
                    if goals_count > 0:
                        if goals_count == 1:
                            subtitle_parts.append(f"{goals_count} gol")
                        else:
                            subtitle_parts.append(f"{goals_count} goles")
                    else:
                        if len(shots_data) == 1:
                            subtitle_parts.append(f"{len(shots_data)} tiro")
                        else:
                            subtitle_parts.append(f"{len(shots_data)} tiros")
                subtitle = "| ".join(subtitle_parts)
            else:
                # Solo rating si no hay stats
                subtitle = f"Rating: {match_info['rating']:.1f}"  # ✅ SIN EVENTOS
            
            self._draw_three_line_header(
                ax,
                context=match_info['context'],
                rating=match_info['rating'],
                stats_line=subtitle,
                show_rating=True
            )
        
        # Ocultar ejes no utilizados
        for i in range(len(recent_matches), len(axs)):
            axs[i].set_visible(False)
        
        # Título principal con estadísticas totales específicas
        category_description = category.title()
        if category == 'attacking':
            category_description = "Atacantes (Regates, Cruces, Conducciones)"
        elif category == 'defensive':
            category_description = "Defensivos (Entradas, Intercepciones, Duelos, Recuperaciones)"
        
        title = f"{self.current_player} - Mapas de Calor: {category_description}"
        
        if category == 'total' and show_stats and total_stats:
            # ESTADÍSTICAS ESPECÍFICAS Y CLARAS
            stats_parts = [f"Promedio: {total_stats['avg_rating']:.1f} rating"]
            stats_parts.append(f"{total_stats['total_passes']} pases ({total_stats['avg_pass_accuracy']:.0f}%)")
            
            if total_stats['total_key_passes'] > 0:
                stats_parts.append(f"{total_stats['total_key_passes']} pases clave")
            if total_stats['total_assists'] > 0:
                stats_parts.append(f"{total_stats['total_assists']} asistencias")
            if total_stats['total_final_third'] > 0:
                stats_parts.append(f"{total_stats['total_final_third']} al tercio final")
            if total_stats['total_shots'] > 0:
                if total_stats['total_goals'] > 0:
                    stats_parts.append(f"{total_stats['total_goals']} goles")
                else:
                    stats_parts.append(f"{total_stats['total_shots']} tiros")
            
            title += f"\n{' | '.join(stats_parts)}"
        
        # Ajustar posición del título según layout
        title_y = 0.95 if layout == '3x2' else 0.92
        fig.suptitle(title, color=self.line_color, fontsize=14, fontweight='bold', y=title_y)
        fig.patch.set_facecolor(self.pitch_color)
        
        if save_images:
            safe_name = self.current_player.replace(" ", "_").replace(".", "")
            layout_suffix = "_instagram" if layout == '3x2' else ""
            filename = f"{safe_name}_{category}_heatmaps_clean{layout_suffix}.png"
            out_path = self.output_dir / filename
            plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=self.pitch_color)
            print(f"Guardado: {out_path}")
        
        plt.show()
        return fig, axs
    
    def plot_enhanced_passes_analysis(self, n_matches=6, save_images=True, save_individual=True, layout='2x3'):
        """
        Análisis completo de pases con estadísticas detalladas y leyendas individuales.
        """
        recent_matches = self.get_player_recent_matches(n_matches)
        
        if len(recent_matches) == 0:
            print("No se encontraron partidos")
            return None, None
        
        # Configurar grid según layout elegido
        n_matches_actual = len(recent_matches)
        
        if layout == '3x2':  # Instagram format
            if n_matches_actual <= 2:
                nrows, ncols = 1, n_matches_actual
                figsize = (6*ncols, 8)
            elif n_matches_actual <= 4:
                nrows, ncols = 2, 2
                figsize = (6*ncols, 8*nrows)
            else:  # 5 o 6 partidos
                nrows, ncols = 3, 2
                figsize = (6*ncols, 8*nrows)
        else:  # layout == '2x3' (original)
            if n_matches_actual <= 3:
                nrows, ncols = 1, n_matches_actual
                figsize = (6*ncols, 7)
            else:
                nrows = 2
                ncols = 3 if n_matches_actual == 6 else min(3, (n_matches_actual + 1) // 2)
                figsize = (6*ncols, 7*nrows)
        
        pitch = Pitch(pitch_type='opta', pitch_color=self.pitch_color, line_color=self.line_color)
        fig, axs = pitch.draw(nrows=nrows, ncols=ncols, figsize=figsize, tight_layout=False)
        
        # Espaciado según layout
        if layout == '3x2':
            plt.subplots_adjust(hspace=0.04, wspace=0.08)  # Distnaincia vertical para campos
        else:
            plt.subplots_adjust(hspace=0.02, wspace=0.05)
        
        if n_matches_actual == 1:
            axs = [axs]
        elif nrows == 1:
            axs = axs if isinstance(axs, list) else [axs]
        else:
            axs = axs.flatten()
        
        arrangement = self._arrange_subplots_chronologically(recent_matches, nrows, ncols)
        
        # Calcular estadísticas totales de pases
        all_match_pass_stats = []
        for match_info in recent_matches:
            match_pass_stats = self._calculate_detailed_pass_stats(match_info['passes_data'])
            all_match_pass_stats.append(match_pass_stats)
        
        total_pass_stats = {
            'total_passes': sum(ps['total_passes'] for ps in all_match_pass_stats),
            'successful_passes': sum(ps['successful_passes'] for ps in all_match_pass_stats),
            'key_passes': sum(ps['key_passes'] for ps in all_match_pass_stats),
            'assists': sum(ps['assists'] for ps in all_match_pass_stats),
            'final_third_passes': sum(ps['final_third_passes'] for ps in all_match_pass_stats),
            'progressive_passes': sum(ps['progressive_passes'] for ps in all_match_pass_stats),
            'crosses': sum(ps['crosses'] for ps in all_match_pass_stats),
            'long_passes': sum(ps['long_passes'] for ps in all_match_pass_stats),
            'medium_passes': sum(ps['medium_passes'] for ps in all_match_pass_stats),
            'short_passes': sum(ps['short_passes'] for ps in all_match_pass_stats),
            'avg_accuracy': sum(ps['pass_accuracy'] for ps in all_match_pass_stats) / len(all_match_pass_stats) if all_match_pass_stats else 0
        }

        for subplot_i, subplot_j, match_idx in arrangement:
            if nrows == 1:
                ax = axs[subplot_j]
            else:
                ax = axs[subplot_i * ncols + subplot_j]
            
            match_info = recent_matches[match_idx]
            passes_data = match_info['passes_data']
            
            if len(passes_data) == 0:
                ax.text(50, 50, "Sin datos\nde pases", ha='center', va='center',
                       color=self.line_color, fontsize=10)
                continue
            
            successful_passes = passes_data[passes_data['pass_outcome'] == 'Successful']
            failed_passes = passes_data[passes_data['pass_outcome'] == 'Unsuccessful']
            key_passes = passes_data[passes_data['is_key_pass'] == True]
            assists = passes_data[passes_data['is_assist'] == True]
            final_third_passes = passes_data[passes_data['to_final_third'] == True]
            
            # LÍNEAS MÁS DELGADAS - Pases por longitud
            if 'q_length' in passes_data.columns and not passes_data['q_length'].isna().all():
                short_passes = passes_data[passes_data['q_length'] <= 15]
                medium_passes = passes_data[(passes_data['q_length'] > 15) & (passes_data['q_length'] <= 35)]
                long_passes = passes_data[passes_data['q_length'] > 35]
                
                for pass_group, width, alpha in [(short_passes, 0.8, 0.3), (medium_passes, 1.2, 0.5), (long_passes, 1.8, 0.6)]:
                    successful = pass_group[pass_group['pass_outcome'] == 'Successful']
                    failed = pass_group[pass_group['pass_outcome'] == 'Unsuccessful']
                    
                    if len(successful) > 0:
                        pitch.arrows(successful['x'], successful['y'],
                                   successful['endX'], successful['endY'],
                                   width=width, headwidth=3+width, headlength=3+width,
                                   color=self.success_color, ax=ax, alpha=alpha)
                    
                    if len(failed) > 0:
                        pitch.arrows(failed['x'], failed['y'],
                                   failed['endX'], failed['endY'],
                                   width=max(0.5, width-0.3), headwidth=2+width*0.5, headlength=2+width*0.5,
                                   color=self.failure_color, ax=ax, alpha=alpha*0.7)
            else:
                # Fallback: pases básicos más delgados
                if len(successful_passes) > 0:
                    pitch.arrows(successful_passes['x'], successful_passes['y'],
                               successful_passes['endX'], successful_passes['endY'],
                               width=1.2, headwidth=4, headlength=4,
                               color=self.success_color, ax=ax, alpha=0.5)
                
                if len(failed_passes) > 0:
                    pitch.arrows(failed_passes['x'], failed_passes['y'],
                               failed_passes['endX'], failed_passes['endY'],
                               width=0.8, headwidth=3, headlength=3,
                               color=self.failure_color, ax=ax, alpha=0.4)
            
            # Pases especiales con líneas delgadas
            if len(final_third_passes) > 0:
                pitch.arrows(final_third_passes['x'], final_third_passes['y'],
                           final_third_passes['endX'], final_third_passes['endY'],
                           width=1.8, headwidth=5, headlength=4,
                           color=self.final_third_color, ax=ax, alpha=0.7)
            
            if len(key_passes) > 0:
                pitch.arrows(key_passes['x'], key_passes['y'],
                           key_passes['endX'], key_passes['endY'],
                           width=2.2, headwidth=6, headlength=5,
                           color=self.key_pass_color, ax=ax, alpha=0.8)
            
            if len(assists) > 0:
                pitch.arrows(assists['x'], assists['y'],
                           assists['endX'], assists['endY'],
                           width=2.8, headwidth=7, headlength=6,
                           color=self.assist_color, ax=ax, alpha=0.9)
            
            # === 1. Calcular estadísticas y construir subtitle ===
            pass_stats = self._calculate_detailed_pass_stats(passes_data)
            subtitle_parts = []
            subtitle_parts.append(f"Pases: {pass_stats['successful_passes']}/{pass_stats['total_passes']} ({pass_stats['pass_accuracy']:.0f}%)")
            if pass_stats['key_passes'] > 0:
                if pass_stats['key_passes'] == 1:
                    subtitle_parts.append(f"{pass_stats['key_passes']} pase clave")
                else:
                    subtitle_parts.append(f"{pass_stats['key_passes']} pases clave")
            if pass_stats['assists'] > 0:
                if pass_stats['assists'] == 1:
                    subtitle_parts.append(f"{pass_stats['assists']} asistencia")
                else:
                    subtitle_parts.append(f"{pass_stats['assists']} asistencias")
            if pass_stats['final_third_passes'] > 0:
                if pass_stats['final_third_passes'] == 1:
                    subtitle_parts.append(f"{pass_stats['final_third_passes']} al tercio final")
                else:
                    subtitle_parts.append(f"{pass_stats['final_third_passes']} al tercio final")
            if pass_stats['progressive_passes'] > 0:
                if pass_stats['progressive_passes'] == 1:
                    subtitle_parts.append(f"{pass_stats['progressive_passes']} pase progresivo")
                else:
                    subtitle_parts.append(f"{pass_stats['progressive_passes']} pases progresivos")
            if pass_stats['crosses'] > 0:
                if pass_stats['crosses'] == 1:
                    subtitle_parts.append(f"{pass_stats['crosses']} centro")
                else:
                    subtitle_parts.append(f"{pass_stats['crosses']} centros")
            subtitle = " | ".join(subtitle_parts)

            # === 2. Dibujar el encabezado de 3 líneas ===
            self._draw_three_line_header(
                ax,
                context=match_info['context'],
                rating=match_info['rating'],
                stats_line=subtitle,
                show_rating=True
            )

            # GUARDADO INDIVIDUAL CON LEYENDA Y NOMBRE DEL JUGADOR
            if save_individual:
                individual_pitch = Pitch(pitch_type='opta', pitch_color=self.pitch_color, line_color=self.line_color)
                individual_fig, individual_ax = individual_pitch.draw(figsize=(10, 7))
                
                # Repetir el dibujo de pases
                if 'q_length' in passes_data.columns and not passes_data['q_length'].isna().all():
                    short_passes = passes_data[passes_data['q_length'] <= 15]
                    medium_passes = passes_data[(passes_data['q_length'] > 15) & (passes_data['q_length'] <= 35)]
                    long_passes = passes_data[passes_data['q_length'] > 35]
                    
                    for pass_group, width, alpha in [(short_passes, 0.8, 0.3), (medium_passes, 1.2, 0.5), (long_passes, 1.8, 0.6)]:
                        successful = pass_group[pass_group['pass_outcome'] == 'Successful']
                        failed = pass_group[pass_group['pass_outcome'] == 'Unsuccessful']
                        
                        if len(successful) > 0:
                            individual_pitch.arrows(successful['x'], successful['y'],
                                       successful['endX'], successful['endY'],
                                       width=width, headwidth=3+width, headlength=3+width,
                                       color=self.success_color, ax=individual_ax, alpha=alpha)
                        
                        if len(failed) > 0:
                            individual_pitch.arrows(failed['x'], failed['y'],
                                       failed['endX'], failed['endY'],
                                       width=max(0.5, width-0.3), headwidth=2+width*0.5, headlength=2+width*0.5,
                                       color=self.failure_color, ax=individual_ax, alpha=alpha*0.7)
                
                if len(final_third_passes) > 0:
                    individual_pitch.arrows(final_third_passes['x'], final_third_passes['y'],
                               final_third_passes['endX'], final_third_passes['endY'],
                               width=1.8, headwidth=5, headlength=4,
                               color=self.final_third_color, ax=individual_ax, alpha=0.7)
                
                if len(key_passes) > 0:
                    individual_pitch.arrows(key_passes['x'], key_passes['y'],
                               key_passes['endX'], key_passes['endY'],
                               width=2.2, headwidth=6, headlength=5,
                               color=self.key_pass_color, ax=individual_ax, alpha=0.8)
                
                if len(assists) > 0:
                    individual_pitch.arrows(assists['x'], assists['y'],
                               assists['endX'], assists['endY'],
                               width=2.8, headwidth=7, headlength=6,
                               color=self.assist_color, ax=individual_ax, alpha=0.9)
                
                # TÍTULO CON NOMBRE DEL JUGADOR para archivo individual
                score_text = f"{match_info['context']['home_goals']}-{match_info['context']['away_goals']}"
                individual_title = f"{self.current_player} - Análisis de Pases"
                individual_subtitle = f"{match_info['context']['home_team']['name']} {score_text} {match_info['context']['away_team']['name']}\n{subtitle}"
                
                individual_fig.suptitle(individual_title, color=self.line_color, fontsize=16, fontweight='bold', y=0.95)
                individual_ax.text(0.5, 1.05, individual_subtitle, transform=individual_ax.transAxes,
                                  ha='center', va='bottom', color=self.line_color, fontsize=10)
                
                # LEYENDA INDIVIDUAL
                from matplotlib.lines import Line2D
                legend_elements = [
                    Line2D([0], [0], color=self.success_color, lw=2, label='Pases exitosos'),
                    Line2D([0], [0], color=self.failure_color, lw=2, label='Pases fallidos'),
                    Line2D([0], [0], color=self.final_third_color, lw=2.5, label='Al tercio final'),
                    Line2D([0], [0], color=self.key_pass_color, lw=3, label='Pases clave'),
                    Line2D([0], [0], color=self.assist_color, lw=3.5, label='Asistencias')
                ]
                individual_fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.98, 0.75),
                                    facecolor=self.pitch_color, edgecolor='none', labelcolor=self.line_color)
                
                individual_fig.patch.set_facecolor(self.pitch_color)
                
                # Guardar archivo individual
                safe_name = self.current_player.replace(" ", "_").replace(".", "")
                match_id_short = str(match_info['match_id'])[-4:]
                individual_filename = f"{safe_name}_pases_partido_{match_id_short}.png"
                individual_path = self.output_dir / individual_filename
                individual_fig.savefig(individual_path, dpi=300, bbox_inches='tight', facecolor=self.pitch_color)
                plt.close(individual_fig)

        # Ocultar ejes no utilizados
        for i in range(len(recent_matches), len(axs)):
            axs[i].set_visible(False)
        
        # Título principal con estadísticas totales detalladas
        title_parts = [f"{self.current_player} - Análisis Completo de Pases"]
        
        stats_parts = []
        stats_parts.append(f"Total: {total_pass_stats['successful_passes']}/{total_pass_stats['total_passes']} pases ({total_pass_stats['avg_accuracy']:.0f}%)")
        
        if total_pass_stats['key_passes'] > 0:
            stats_parts.append(f"{total_pass_stats['key_passes']} pases clave")
        if total_pass_stats['assists'] > 0:
            stats_parts.append(f"{total_pass_stats['assists']} asistencias")
        if total_pass_stats['final_third_passes'] > 0:
            stats_parts.append(f"{total_pass_stats['final_third_passes']} al tercio final")
        if total_pass_stats['progressive_passes'] > 0:
            stats_parts.append(f"{total_pass_stats['progressive_passes']} progresivos")
        if total_pass_stats['crosses'] > 0:
            stats_parts.append(f"{total_pass_stats['crosses']} centros")
        
        title_parts.append(' | '.join(stats_parts))
        title = '\n'.join(title_parts)
        
        # Ajustar posición del título según layout
        title_y = 0.92 if layout == '3x2' else 0.90
        fig.suptitle(title, color=self.line_color, fontsize=14, fontweight='bold', y=title_y)
        fig.patch.set_facecolor(self.pitch_color)
        
        # Leyenda principal
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color=self.success_color, lw=1.5, label='Exitosos'),
            Line2D([0], [0], color=self.failure_color, lw=1.5, label='Fallidos'),
            Line2D([0], [0], color=self.final_third_color, lw=2, label='Al tercio final'),
            Line2D([0], [0], color=self.key_pass_color, lw=2.5, label='Pases clave'),
            Line2D([0], [0], color=self.assist_color, lw=3, label='Asistencias')
        ]
        # Ajustar posición de leyenda según layout
        legend_y = 0.90 if layout == '3x2' else 0.90
        fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.98, legend_y),
                facecolor=self.pitch_color, edgecolor='none', labelcolor=self.line_color)
        
        if save_images:
            safe_name = self.current_player.replace(" ", "_").replace(".", "")
            filename = f"{safe_name}_pases_completo.png"
            out_path = self.output_dir / filename
            plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=self.pitch_color)
            print(f"Guardado conjunto: {out_path}")
            if save_individual:
                print(f"Guardados {len(recent_matches)} archivos individuales con leyendas")
        
        plt.show()
        return fig, axs
    
    def plot_clean_shots_analysis(self, n_matches=6, save_images=True):
        """
        Análisis limpio de tiros sin escudos.
        """
        recent_matches = self.get_player_recent_matches(n_matches)
        
        if len(recent_matches) == 0:
            print("No se encontraron partidos")
            return None, None
        
        # Grid más compacto
        n_matches_actual = len(recent_matches)
        if n_matches_actual <= 3:
            nrows, ncols = 1, n_matches_actual
            figsize = (5*ncols, 4)
        else:
            nrows = 2
            ncols = 3 if n_matches_actual == 6 else min(3, (n_matches_actual + 1) // 2)
            figsize = (5*ncols, 4*nrows)
        
        pitch = Pitch(pitch_type='opta', half=True, pitch_color=self.pitch_color, line_color=self.line_color)
        fig, axs = pitch.draw(nrows=nrows, ncols=ncols, figsize=figsize, tight_layout=False)
        
        plt.subplots_adjust(hspace=0.15, wspace=0.01)
        
        if n_matches_actual == 1:
            axs = [axs]
        elif nrows == 1:
            axs = axs if isinstance(axs, list) else [axs]
        else:
            axs = axs.flatten()
        
        total_goals = 0
        total_shots_all_matches = 0
        total_on_target = 0
        
        arrangement = self._arrange_subplots_chronologically(recent_matches, nrows, ncols)
        
        for subplot_i, subplot_j, match_idx in arrangement:
            if nrows == 1:
                ax = axs[subplot_j]
            else:
                ax = axs[subplot_i * ncols + subplot_j]
            
            match_info = recent_matches[match_idx]
            shots_data = match_info['shots_data']
            
            shots_data = shots_data[shots_data['x'] >= 50] if len(shots_data) > 0 else shots_data
            
            if len(shots_data) == 0:
                ax.text(75, 50, "Sin tiros", ha='center', va='center',
                       color=self.line_color, fontsize=10)
            else:
                outcome_col = 'shot_outcome_clean' if 'shot_outcome_clean' in shots_data.columns else 'shot_outcome'
                
                goals = shots_data[shots_data[outcome_col] == 'Goal']
                saved = shots_data[shots_data[outcome_col] == 'Saved']
                missed = shots_data[shots_data[outcome_col] == 'Missed']
                blocked = shots_data[shots_data[outcome_col] == 'Blocked']
                post = shots_data[shots_data[outcome_col] == 'Post']
                
                total_goals += len(goals)
                total_shots_all_matches += len(shots_data)
                total_on_target += len(goals) + len(saved)
                
                # Marcadores ligeramente más pequeños
                if len(goals) > 0:
                    pitch.scatter(goals['x'], goals['y'], ax=ax, s=350, c=self.goal_color, 
                                 marker='*', edgecolors='black', linewidth=2.5, alpha=1.0,
                                 zorder=5)
                
                if len(saved) > 0:
                    pitch.scatter(saved['x'], saved['y'], ax=ax, s=180, 
                                 c=self.success_color, marker='o', edgecolors='white', 
                                 linewidth=1.5, alpha=0.8)
                
                if len(missed) > 0:
                    pitch.scatter(missed['x'], missed['y'], ax=ax, s=130, 
                                 c=self.failure_color, marker='x', linewidth=2.5,
                                 alpha=0.7)
                
                if len(blocked) > 0:
                    pitch.scatter(blocked['x'], blocked['y'], ax=ax, s=150, 
                                 c='orange', marker='s', edgecolors='white',
                                 linewidth=1.5, alpha=0.8)
                
                if len(post) > 0:
                    pitch.scatter(post['x'], post['y'], ax=ax, s=170, 
                                 c='purple', marker='D', edgecolors='white',
                                 linewidth=1.5, alpha=0.8)
            
            # Header limpio
            score_text = f"{match_info['context']['home_goals']}-{match_info['context']['away_goals']}"
            self._draw_three_line_header(ax, match_info['context'], match_info['team_id'], score_text)
            
            # Subtítulo con métricas
            if len(shots_data) > 0:
                outcome_col = 'shot_outcome_clean' if 'shot_outcome_clean' in shots_data.columns else 'shot_outcome'
                goals_count = len(shots_data[shots_data[outcome_col] == 'Goal'])
                saved_count = len(shots_data[shots_data[outcome_col] == 'Saved'])
                on_target_count = goals_count + saved_count
                subtitle = f"Tiros: {len(shots_data)} ({on_target_count} a portería)"
                if goals_count > 0: 
                    subtitle += f" | {goals_count} goles"
            else:
                subtitle = "Sin tiros registrados"
            
            self._draw_subtitle_line(ax, subtitle)
        
        # Ocultar ejes no utilizados
        for i in range(len(recent_matches), len(axs)):
            axs[i].set_visible(False)
        
        # Estadísticas totales
        on_target_percentage = (total_on_target / total_shots_all_matches * 100) if total_shots_all_matches > 0 else 0
        
        title = (f"{self.current_player} - Análisis Completo de Tiros\n"
                f"Total: {total_goals} goles en {total_shots_all_matches} tiros "
                f"({total_on_target} a portería - {on_target_percentage:.0f}%)")
        
        fig.suptitle(title, color=self.line_color, fontsize=14, fontweight='bold', y=0.90)
        fig.patch.set_facecolor(self.pitch_color)
        
        # Leyenda
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='*', color='w', markerfacecolor=self.goal_color, 
                   markersize=12, label='Goles', markeredgecolor='black', markeredgewidth=1.5),
            Line2D([0], [0], marker='o', color='w', markerfacecolor=self.success_color, 
                   markersize=10, label='A portería', markeredgecolor='white', markeredgewidth=1),
            Line2D([0], [0], marker='x', color=self.failure_color, 
                   markersize=10, label='Fuera', linestyle='None', markeredgewidth=2),
            Line2D([0], [0], marker='s', color='w', markerfacecolor='orange', 
                   markersize=8, label='Bloqueado', markeredgecolor='white', markeredgewidth=1),
            Line2D([0], [0], marker='D', color='w', markerfacecolor='purple', 
                   markersize=8, label='Poste', markeredgecolor='white', markeredgewidth=1)
        ]
        fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.98, 0.88),
                  facecolor=self.pitch_color, edgecolor='none', labelcolor=self.line_color)
        
        if save_images:
            safe_name = self.current_player.replace(" ", "_").replace(".", "")
            filename = f"{safe_name}_tiros_limpio.png"
            out_path = self.output_dir / filename
            plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=self.pitch_color)
            print(f"Guardado: {out_path}")
        
        plt.show()
        return fig, axs
    
    def generate_clean_complete_analysis(self, n_matches=6, save_images=True, layout='2x3'):
        """Genera análisis completo limpio con todas las mejoras solicitadas."""
        if not self.current_player:
            raise ValueError("Debe establecer un jugador primero")
        
        print(f"Generando análisis limpio para {self.current_player}")
        print("=" * 60)
        
        # 1. Solo mapas de calor esenciales
        categories = ['total', 'passes', 'defensive', 'attacking']
        for i, category in enumerate(categories, 1):
            print(f"{i}. Mapas de calor: {category}...")
            self.plot_clean_heatmaps_by_category(category, n_matches, save_images, layout=layout)
        
        # 2. Análisis completo de pases con guardado individual
        print("5. Análisis completo de pases...")
        self.plot_enhanced_passes_analysis(n_matches, save_images, save_individual=True, layout=layout)
        
        # 3. Análisis de tiros
        print("6. Análisis de tiros...")
        self.plot_clean_shots_analysis(n_matches, save_images)
        
        print("\n✅ Análisis limpio completado!")
        print("✅ Mejoras implementadas:")
        print("  • Sin escudos - diseño limpio")
        print("  • Espaciado muy reducido (hspace=0.08, wspace=0.05)")
        print("  • Estadísticas específicas ('3 pases clave' en lugar de '3 clave')")
        print("  • Mayor separación entre título y subtítulo")
        print("  • Solo mapa defensivo unificado")
        print("  • Estadísticas detalladas de tipos de pases")
        print("  • Leyendas en archivos individuales")
        print("  • Nombre del jugador en visualizaciones individuales")
        
        return True
    
# =====================================================================
# FUNCIONES DE USO DIRECTO LIMPIAS
# =====================================================================

def create_clean_analyzer(base_path, team_identity_path=None):
    """Crea analizador limpio optimizado."""
    if team_identity_path is None:
        team_identity_path = Path(base_path).parent.parent / 'dictionaries' / 'team_identity.csv'
    
    analyzer = RefinedPlayerAnalyzer(base_path, team_identity_path)
    return analyzer

def clean_complete_analysis(base_path, player_name, max_matches_scan=15):
    """Análisis completo limpio de un jugador específico."""
    team_path = Path(base_path).parent.parent / 'dictionaries' / 'team_identity.csv'
    
    analyzer = create_clean_analyzer(base_path, team_path)
    print("Escaneando partidos...")
    analyzer.scan_all_matches(max_matches_scan)
    analyzer.set_player(player_name)
    
    return analyzer.generate_clean_complete_analysis()

# =====================================================================
# CONFIGURACIÓN FINAL LIMPIA
# =====================================================================

BASE_PATH = r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\matchcenter\MatchCenter\Competition\Season"
TEAM_IDENTITY_PATH = r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\dictionaries\team_identity.csv"

print("SISTEMA DE ANÁLISIS VISUAL LIMPIO")
print("=" * 50)
print("Mejoras implementadas:")
print("  • Sin escudos - diseño completamente limpio")
print("  • Espaciado ultra-compacto")
print("  • Estadísticas específicas y claras")
print("  • Mayor separación título-subtítulo")
print("  • Solo 4 mapas de calor: total, passes, defensive, attacking")
print("  • Mapa defensivo unificado con todas las acciones")
print("  • Análisis de pases con estadísticas detalladas")
print("  • Archivos individuales con leyendas y nombre del jugador")
print("\nUso recomendado:")
print("analyzer = create_clean_analyzer(BASE_PATH, TEAM_IDENTITY_PATH)")
print("analyzer.scan_all_matches(15)")
print("analyzer.set_player('Pablo Fornals')")
print("analyzer.generate_clean_complete_analysis()")                

In [ ]:
# Crear tu analyzer como siempre
analyzer = create_clean_analyzer(BASE_PATH, TEAM_IDENTITY_PATH)
analyzer.scan_all_matches(69)
analyzer.set_player('Pedri')

# Formato tradicional (2x3)
# analyzer.generate_clean_complete_analysis(n_matches=6, layout='2x3')

# Formato Instagram (3x2)
# analyzer.generate_clean_complete_analysis(n_matches=6, layout='3x2')

# También funciona individualmente
analyzer.plot_clean_heatmaps_by_category('total', layout='3x2')
analyzer.plot_enhanced_passes_analysis(layout='3x2')